In [ ]:
import pandas as pd
import numpy as np

# Cargar base
df = pd.read_parquet('base_ercue2025.parquet')

# ============================================================
# MERGE CON SECTOR TÍPICO
# ============================================================
st = pd.read_excel('Sector_Tipico.xlsx', sheet_name='Hoja1')
st['Código Sistema'] = st['Código Sistema'].str.strip()
df['sistema'] = df['sistema'].str.strip()

st_simple = st[['Código Sistema', 'Sector']].rename(columns={'Código Sistema': 'sistema', 'Sector': 'sector_tipico'})
st_simple['sector_tipico'] = st_simple['sector_tipico'].astype(str).replace({'1': 'ST1', '2': 'ST2', '3': 'ST3', '4': 'ST4'})

df = df.merge(st_simple, on='sistema', how='left')

# Corregir los sin match
df.loc[df['sistema'] == 'SR0235', 'sector_tipico'] = 'ST3'
for s in ['SR007', 'SR0107', 'SR0109', 'SR0115', 'SR593', 'SR684']:
    df.loc[df['sistema'] == s, 'sector_tipico'] = 'SER'

print("=== SECTOR TÍPICO ===")
print(df['sector_tipico'].value_counts())
print(f"Sin sector: {df['sector_tipico'].isna().sum()}")

# ============================================================
# ENERGY BURDEN Y UMBRAL 2M
# ============================================================
gasto_total_mes = df['ga_aj_total_pc'] * df['N_PERSONAS']
df['energy_burden'] = df['gasto_mes_electricidad'] / gasto_total_mes

mask = df['energy_burden'].notna() & (df['energy_burden'] > 0) & (df['energy_burden'] < 1)
eb_valid = df.loc[mask, 'energy_burden']
weights = df.loc[mask, 'Fact_exp_hog']

sorted_idx = np.argsort(eb_valid.values)
sorted_eb = eb_valid.values[sorted_idx]
sorted_w = weights.values[sorted_idx]
cumsum = np.cumsum(sorted_w)
cutoff = sorted_w.sum() / 2.0
mediana_pond = sorted_eb[cumsum >= cutoff][0]
umbral_2m = mediana_pond * 2

df['sobrecarga_2m'] = (df['energy_burden'] > umbral_2m).astype(float)
df.loc[df['energy_burden'].isna(), 'sobrecarga_2m'] = np.nan

print(f"\nMediana ponderada: {mediana_pond:.4f} ({mediana_pond*100:.3f}%)")
print(f"Umbral 2M: {umbral_2m:.4f} ({umbral_2m*100:.2f}%)")
print(f"Hogares en sobrecarga: {df['sobrecarga_2m'].sum():.0f} ({df['sobrecarga_2m'].mean()*100:.1f}%)")

# ============================================================
# SEIN vs AISLADO (por % de descuento observado)
# ============================================================
df['pct_descuento_b1'] = 1 - (df['fose30'] / df['mcter0'])

def clasificar_sein(row):
    st = row['sector_tipico']
    desc = row['pct_descuento_b1']
    if pd.isna(st) or pd.isna(desc):
        return None
    if st in ['ST1', 'ST2']:
        return 'Aislado' if desc >= 0.45 else 'SEIN'
    elif st in ['ST3', 'ST4', 'SER']:
        if desc >= 0.79:
            return 'Fotovoltaico'
        elif desc >= 0.68:
            return 'Aislado'
        else:
            return 'SEIN'
    return None

df['tipo_sistema'] = df.apply(clasificar_sein, axis=1)

print(f"\n=== SEIN vs AISLADO ===")
print(df['tipo_sistema'].value_counts())
print(f"\nPor sector típico:")
print(pd.crosstab(df['sector_tipico'], df['tipo_sistema']))

# ============================================================
# GUARDAR
# ============================================================
df.to_parquet('base_ercue2025_st.parquet', index=False)
print("\nGuardado como base_ercue2025_st.parquet")

In [ ]:
# Ver qué valores tiene p5_6_2_1
print(df['p5_6_2_1'].value_counts().head(20))
print(f"\nTipo: {df['p5_6_2_1'].dtype}")
print(f"\nValores que parecen nulos:")
print(f"  Vacío '': {(df['p5_6_2_1'] == '').sum()}")
print(f"  'nan': {(df['p5_6_2_1'] == 'nan').sum()}")
print(f"  'None': {(df['p5_6_2_1'] == 'None').sum()}")
print(f"  NaN real: {df['p5_6_2_1'].isna().sum()}")
print(f"  Largo <= 2: {(df['p5_6_2_1'].str.len() <= 2).sum()}")

In [ ]:
import numpy as np

# Elegibilidad FOSE: consumo <= 140 kWh. Los que no tienen dato de consumo quedan como NaN (no se clasifican)
df['elegible'] = np.where(df['consumo_fose'].isna(), np.nan, df['consumo_fose'] <= 140)

# Condición de pobreza: pobre extremo o pobre no extremo según clasificación INEI
df['pobre'] = df['POBREZA'].isin(['Pobre no extremo', 'Pobre extremo'])

In [ ]:
import numpy as np
import pandas as pd

# ============================================================
# PASO 1: MULTIPREDIO
# ============================================================
# La variable p5_6_2_1 es el número de suministro eléctrico.
# Si dos hogares comparten el mismo número = multipredio.
#
# Problema detectado: 3.946 registros tienen un espacio en
# blanco (' ') en vez de número real. Esos NO son multipredio,
# son datos faltantes. También hay 47 que dicen "NO SE VE".
# Si no los filtramos, el multipredio se infla al 30%.
# ============================================================

print("=== SUMINISTROS INVÁLIDOS ===")
print(f"Espacio en blanco: {(df['p5_6_2_1'].str.strip().str.len() == 0).sum()}")
print(f"NO SE VE: {(df['p5_6_2_1'] == 'NO SE VE').sum()}")

# Filtro: válido = largo > 2 caracteres y no es "NO SE VE"
mask_valido = (df['p5_6_2_1'].str.strip().str.len() > 2) & (df['p5_6_2_1'] != 'NO SE VE')
print(f"Suministros válidos: {mask_valido.sum()}")

# Multipredio: suministros que aparecen más de una vez
df['multipredio'] = False
duplicados = df.loc[mask_valido, 'p5_6_2_1'].duplicated(keep=False)
df.loc[mask_valido, 'multipredio'] = duplicados.values
print(f"Multipredio: {df['multipredio'].sum()} ({df['multipredio'].sum()/mask_valido.sum()*100:.1f}%)")

# ============================================================
# PASO 2: ELEGIBILIDAD FOSE
# ============================================================
# Elegibl

In [ ]:
# ============================================================
# CELDA 2: EXCLUSIÓN POR ESTRATO ALTO EN LIMA (ST1)
# ============================================================
# La Ley 31429 excluye del FOSE a hogares de estrato alto
# en Lima con consumo entre 100-140 kWh.
# La ERCUE no tiene estrato de manzana del INEI.
# Proxy: conglomerados de Lima cuyo gasto per cápita promedio
# supera el percentil 80 (~20% más rico de Lima).
# Esto replica la proporción de manzanas de estrato alto
# y medio-alto según los planos estratificados del INEI.
# Limitación: cada conglomerado tiene ~5 hogares, hay ruido.
# ============================================================

lima = df[df['sector_tipico'] == 'ST1'].copy()

# Gasto promedio por conglomerado en Lima
gasto_congl = lima.groupby('conglomerado_M')['ga_aj_total_pc'].mean()

# Percentil 80 = umbral de estrato alto
p80 = gasto_congl.quantile(0.80)
print(f"Percentil 80 gasto per cápita Lima: S/{p80:.0f}")

# Conglomerados clasificados como estrato alto
congl_alto = gasto_congl[gasto_congl > p80].index

# Hogares de estrato alto con consumo 100-140 kWh → excluidos del FOSE
mask_estrato = (
    (df['sector_tipico'] == 'ST1') &
    (df['conglomerado_M'].isin(congl_alto)) &
    (df['consumo_fose'] >= 100) &
    (df['consumo_fose'] <= 140)
)
print(f"Hogares excluidos por estrato alto en Lima: {mask_estrato.sum()}")

# Marcar como no elegibles
df.loc[mask_estrato, 'elegible'] = 0

In [ ]:
# ¿Cuántos pobres excluidos tienen consumo_fose nulo?
st1_pobres_excl = st1_pobres[st1_pobres['elegible'] == False]
print(f"Pobres excluidos ST1: {len(st1_pobres_excl)}")
print(f"  Con consumo_fose: {st1_pobres_excl['consumo_fose'].notna().sum()}")
print(f"  Sin consumo_fose (NaN): {st1_pobres_excl['consumo_fose'].isna().sum()}")

# ¿Los NaN en consumo_fose se clasifican como elegible=False?
print(f"\n¿Cómo se clasifican los NaN en consumo_fose?")
nulos = df[df['consumo_fose'].isna()]
print(f"Total NaN consumo: {len(nulos)}")
print(f"  elegible=True: {(nulos['elegible'] == True).sum()}")
print(f"  elegible=False: {(nulos['elegible'] == False).sum()}")

In [ ]:
# ============================================================
# CELDA ÚNICA: MULTIPREDIO + ELEGIBILIDAD + ESTRATO ALTO + ERRORES
# ============================================================
import numpy as np

# ------------------------------------------------------------
# PASO 1: MULTIPREDIO
# ------------------------------------------------------------
# La variable p5_6_2_1 es el número de suministro eléctrico.
# Si dos hogares tienen el mismo número = comparten suministro
# = multipredio = excluidos del FOSE.
#
# Problema: 3.946 registros tienen un espacio en blanco (' ')
# y 47 dicen "NO SE VE". Si no los filtramos, el multipredio
# se infla al 30% porque Python los trata como valor repetido.
# Solución: filtrar suministros con largo > 2 caracteres.
# ------------------------------------------------------------

mask_valido = (df['p5_6_2_1'].str.strip().str.len() > 2) & (df['p5_6_2_1'] != 'NO SE VE')

df['multipredio'] = False
duplicados = df.loc[mask_valido, 'p5_6_2_1'].duplicated(keep=False)
df.loc[mask_valido, 'multipredio'] = duplicados.values

print(f"=== MULTIPREDIO ===")
print(f"Suministros válidos: {mask_valido.sum()}")
print(f"Suministros en blanco o NO SE VE: {(~mask_valido).sum()}")
print(f"Multipredio: {df['multipredio'].sum()} ({df['multipredio'].sum()/mask_valido.sum()*100:.1f}%)")

# ------------------------------------------------------------
# PASO 2: ELEGIBILIDAD FOSE
# ------------------------------------------------------------
# Elegible = consumo <= 140 kWh.
# Los 670 hogares sin dato de consumo (NaN) quedan como NaN,
# NO como excluidos. Si no hacemos esto, Python trata
# NaN <= 140 como False y los cuenta como si consumieran
# más de 140 kWh, inflando el error de exclusión del 5% al 10%.
# ------------------------------------------------------------

df['elegible'] = np.where(df['consumo_fose'].isna(), np.nan, df['consumo_fose'] <= 140)
df['pobre'] = df['POBREZA'].isin(['Pobre no extremo', 'Pobre extremo'])

print(f"\n=== ELEGIBILIDAD ===")
print(f"Elegibles (<=140 kWh): {(df['elegible'] == 1).sum()}")
print(f"Excluidos (>140 kWh): {(df['elegible'] == 0).sum()}")
print(f"Sin dato de consumo: {df['elegible'].isna().sum()}")

# ------------------------------------------------------------
# PASO 3: EXCLUSIÓN POR ESTRATO ALTO EN LIMA (ST1)
# ------------------------------------------------------------
# La Ley 31429 excluye del FOSE a hogares de estrato alto
# en Lima con consumo entre 100-140 kWh.
# La ERCUE no tiene estrato de manzana del INEI.
# Proxy: conglomerados de Lima cuyo gasto per cápita promedio
# supera el percentil 80 (~20% más rico de Lima).
# Limitación: cada conglomerado tiene ~5 hogares, hay ruido.
# ------------------------------------------------------------

lima = df[df['sector_tipico'] == 'ST1'].copy()
gasto_congl = lima.groupby('conglomerado_M')['ga_aj_total_pc'].mean()
p80 = gasto_congl.quantile(0.80)
congl_alto = gasto_congl[gasto_congl > p80].index

mask_estrato = (
    (df['sector_tipico'] == 'ST1') &
    (df['conglomerado_M'].isin(congl_alto)) &
    (df['consumo_fose'] >= 100) &
    (df['consumo_fose'] <= 140)
)
df.loc[mask_estrato, 'elegible'] = 0

print(f"\n=== EXCLUSIÓN ESTRATO ALTO LIMA ===")
print(f"Percentil 80 gasto per cápita Lima: S/{p80:.0f}")
print(f"Hogares excluidos por estrato alto: {mask_estrato.sum()}")

# ------------------------------------------------------------
# PASO 4: ERRORES DE FOCALIZACIÓN
# ------------------------------------------------------------
# Solo hogares con consumo válido (no NaN).
# EI = elegibles no pobres / total elegibles
#   → hogares que reciben FOSE sin necesitarlo
# EE = pobres no elegibles / total pobres
#   → hogares pobres sin FOSE (por consumo >140 o estrato alto)
# Multipredio = elegibles con suministro compartido
#   → barrera de acceso, especialmente en SER (rural)
# ------------------------------------------------------------

df_con_consumo = df[df['consumo_fose'].notna()].copy()
elegibles = df_con_consumo[df_con_consumo['elegible'] == 1]
pobres = df_con_consumo[df_con_consumo['pobre'] == True]

ei = (elegibles['pobre'] == False).sum() / len(elegibles) * 100
ee = (pobres['elegible'] == 0).sum() / len(pobres) * 100
multi = elegibles['multipredio'].sum() / len(elegibles) * 100

print(f"\n=== ERRORES GLOBALES ===")
print(f"Error de Inclusión: {ei:.1f}%")
print(f"Error de Exclusión: {ee:.1f}%")
print(f"Multipredio elegibles: {multi:.1f}%")

print(f"\n{'Sector':<8} {'N':>6} {'EI%':>8} {'EE%':>8} {'Multi%':>8}")
print("-" * 42)
for st in ['ST1', 'ST2', 'ST3', 'ST4', 'SER']:
    sub = df_con_consumo[df_con_consumo['sector_tipico'] == st]
    eleg = sub[sub['elegible'] == 1]
    pob = sub[sub['pobre'] == True]
    ei_st = (eleg['pobre'] == False).sum() / len(eleg) * 100 if len(eleg) > 0 else 0
    ee_st = (pob['elegible'] == 0).sum() / len(pob) * 100 if len(pob) > 0 else 0
    multi_st = eleg['multipredio'].sum() / len(eleg) * 100 if len(eleg) > 0 else 0
    print(f"{st:<8} {len(sub):>6} {ei_st:>7.1f}% {ee_st:>7.1f}% {multi_st:>7.1f}%")

df.to_parquet('base_ercue2025_st.parquet', index=False)
print("\nGuardado.")

In [ ]:
import numpy as np
from scipy import stats
from sklearn.linear_model import LogisticRegression

# ============================================================
# CEM + IPW: CAPACIDAD DISCRIMINANTE DEL UMBRAL 140 kWh
# ============================================================
# Pregunta: ¿el umbral de 140 kWh separa hogares con distinto
# nivel de sobrecarga energética?
#
# Variable de tratamiento T:
#   T=1: elegible (consumo ≤ 140 kWh Y no excluido por estrato
#        alto en Lima). Usa la variable 'elegible' que ya tiene
#        ambos filtros aplicados.
#   T=0: no elegible (consumo > 140 kWh O estrato alto Lima)
#
# ¿Por qué NO incluimos multipredio en T?
# Porque el multipredio es una barrera ADMINISTRATIVA (suministro
# compartido), no del UMBRAL. El CEM evalúa si el umbral de
# consumo discrimina bien. Un hogar multipredio con consumo de
# 80 kWh sí es elegible por consumo — que no reciba el FOSE
# por otra razón no cambia su posición respecto al umbral.
# El multipredio se reporta aparte como barrera adicional.
#
# Covariables de emparejamiento:
#   - Sector típico: ST1-ST2 (urbano), ST3-ST4 (semi-rural), SER
#   - Área: Urbano / Rural
#   - Tamaño del hogar: 1-2, 3-4, 5-6, 7+
#
# Outcome: sobrecarga 2M (energy burden > 6,26%)
# ============================================================

# Filtrar hogares con datos completos
df_cem = df[df['consumo_fose'].notna() & df['sector_tipico'].notna() &
            df['sobrecarga_2m'].notna() & df['elegible'].notna()].copy()

# Variable de tratamiento: usa 'elegible' (incluye filtro estrato alto)
df_cem['T'] = df_cem['elegible'].astype(int)

# Categorizar tamaño del hogar
df_cem['tamano_cat'] = pd.cut(df_cem['N_PERSONAS'], bins=[0, 2, 4, 6, 99], labels=['1-2', '3-4', '5-6', '7+'])

# Agrupar sector típico para CEM
df_cem['sector_cem'] = df_cem['sector_tipico'].replace({'ST1': 'ST1-ST2', 'ST2': 'ST1-ST2', 'ST3': 'ST3-ST4', 'ST4': 'ST3-ST4'})

# ============================================================
# CEM por ventana: ±20 kWh (principal) y ±30 kWh (robustez)
# ============================================================
for bw_label, bw_low, bw_high in [('±20', 120, 160), ('±30', 110, 170)]:
    ventana = df_cem[(df_cem['consumo_fose'] >= bw_low) & (df_cem['consumo_fose'] <= bw_high)].copy()

    print(f"\n{'='*60}")
    print(f"VENTANA {bw_label} kWh ({bw_low}-{bw_high})")
    print(f"{'='*60}")
    print(f"Total: {len(ventana)}, T=1: {ventana['T'].sum()}, T=0: {(ventana['T']==0).sum()}")

    # Crear celdas de emparejamiento exacto
    ventana['celda'] = ventana['sector_cem'].astype(str) + '_' + ventana['area'].astype(str) + '_' + ventana['tamano_cat'].astype(str)

    # Solo celdas con al menos 1 tratado y 1 control
    celdas_validas = ventana.groupby('celda')['T'].agg(['sum', 'count'])
    celdas_validas['n_control'] = celdas_validas['count'] - celdas_validas['sum']
    celdas_validas = celdas_validas[(celdas_validas['sum'] > 0) & (celdas_validas['n_control'] > 0)]

    ventana_match = ventana[ventana['celda'].isin(celdas_validas.index)].copy()

    print(f"Celdas válidas: {len(celdas_validas)}")
    print(f"Hogares emparejados: {len(ventana_match)} de {len(ventana)} ({len(ventana_match)/len(ventana)*100:.1f}%)")

    # ATT ponderado por celda
    resultados_celda = []
    for celda in celdas_validas.index:
        sub = ventana_match[ventana_match['celda'] == celda]
        t1 = sub[sub['T'] == 1]['sobrecarga_2m'].mean()
        t0 = sub[sub['T'] == 0]['sobrecarga_2m'].mean()
        n1 = (sub['T'] == 1).sum()
        resultados_celda.append({'celda': celda, 'att': t1 - t0, 'n1': n1})

    res = pd.DataFrame(resultados_celda)
    n_total = res['n1'].sum()
    res['peso'] = res['n1'] / n_total
    att = (res['att'] * res['peso']).sum()

    # Bootstrap SE (2000 réplicas)
    np.random.seed(42)
    att_boot = []
    for _ in range(2000):
        idx = np.random.choice(len(ventana_match), size=len(ventana_match), replace=True)
        boot = ventana_match.iloc[idx]
        boot_res = []
        for celda in celdas_validas.index:
            sub = boot[boot['celda'] == celda]
            t1_vals = sub[sub['T'] == 1]['sobrecarga_2m']
            t0_vals = sub[sub['T'] == 0]['sobrecarga_2m']
            if len(t1_vals) > 0 and len(t0_vals) > 0:
                boot_res.append({'att': t1_vals.mean() - t0_vals.mean(), 'n1': len(t1_vals)})
        if boot_res:
            bdf = pd.DataFrame(boot_res)
            bdf['peso'] = bdf['n1'] / bdf['n1'].sum()
            att_boot.append((bdf['att'] * bdf['peso']).sum())

    se = np.std(att_boot)
    pval = 2 * (1 - stats.norm.cdf(abs(att / se))) if se > 0 else 1
    ci90 = (att - 1.645 * se, att + 1.645 * se)
    ci95 = (att - 1.96 * se, att + 1.96 * se)

    print(f"\nATT = {att*100:.2f} pp")
    print(f"SE  = {se*100:.2f} pp")
    print(f"p   = {pval:.3f}")
    print(f"IC 90%: [{ci90[0]*100:.2f}, {ci90[1]*100:.2f}]")
    print(f"IC 95%: [{ci95[0]*100:.2f}, {ci95[1]*100:.2f}]")

# ============================================================
# IPW ±20 kWh (robustez)
# ============================================================
# Segundo estimador con supuestos distintos al CEM.
# Si ambos dan resultados similares, la conclusión es robusta.
# ============================================================

ventana20 = df_cem[(df_cem['consumo_fose'] >= 120) & (df_cem['consumo_fose'] <= 160)].copy()
ventana20 = ventana20.dropna(subset=['sobrecarga_2m'])

X = pd.get_dummies(ventana20[['sector_cem', 'area', 'tamano_cat']], drop_first=True).astype(float)
y = ventana20['T'].values

lr = LogisticRegression(max_iter=1000)
lr.fit(X, y)
ventana20['ps'] = lr.predict_proba(X)[:, 1]

print(f"\n{'='*60}")
print(f"IPW ±20 kWh")
print(f"{'='*60}")
print(f"PS media elegibles: {ventana20.loc[ventana20['T']==1, 'ps'].mean():.3f}")
print(f"PS media excluidos: {ventana20.loc[ventana20['T']==0, 'ps'].mean():.3f}")

# ATT por IPW
t1 = ventana20[ventana20['T'] == 1]['sobrecarga_2m'].mean()
w = ventana20.loc[ventana20['T'] == 0, 'ps'] / (1 - ventana20.loc[ventana20['T'] == 0, 'ps'])
t0_ipw = np.average(ventana20.loc[ventana20['T'] == 0, 'sobrecarga_2m'], weights=w)
att_ipw = t1 - t0_ipw

# Bootstrap SE para IPW
att_ipw_boot = []
for _ in range(2000):
    idx = np.random.choice(len(ventana20), size=len(ventana20), replace=True)
    boot = ventana20.iloc[idx]
    X_b = pd.get_dummies(boot[['sector_cem', 'area', 'tamano_cat']], drop_first=True).astype(float)
    y_b = boot['T'].values
    try:
        lr_b = LogisticRegression(max_iter=1000)
        lr_b.fit(X_b, y_b)
        ps_b = lr_b.predict_proba(X_b)[:, 1]
        t1_b = boot[boot['T'] == 1]['sobrecarga_2m'].mean()
        mask0 = boot['T'] == 0
        w_b = ps_b[mask0.values] / (1 - ps_b[mask0.values])
        t0_b = np.average(boot.loc[mask0, 'sobrecarga_2m'].values, weights=w_b)
        att_ipw_boot.append(t1_b - t0_b)
    except:
        pass

se_ipw = np.std(att_ipw_boot)
pval_ipw = 2 * (1 - stats.norm.cdf(abs(att_ipw / se_ipw))) if se_ipw > 0 else 1

print(f"ATT IPW = {att_ipw*100:.2f} pp")
print(f"SE  = {se_ipw*100:.2f} pp")
print(f"p   = {pval_ipw:.3f}")

In [ ]:
# ============================================================
# HETEROGENEIDAD POR SECTOR TÍPICO (CEM ±20 kWh)
# ============================================================
# ¿El efecto es distinto según la densidad del sistema eléctrico?
# ST1-ST2: sistemas de alta densidad (Lima y ciudades grandes)
# ST3-ST4: sistemas de densidad intermedia (ciudades menores)
# SER: sistemas eléctricos rurales (baja densidad)
# La clasificación por sector típico refleja la estructura de
# costos del sistema eléctrico, no directamente urbano/rural,
# aunque en la práctica coinciden en buena medida.
# Se estima el CEM por separado para cada grupo.
# ============================================================

ventana20 = df_cem[(df_cem['consumo_fose'] >= 120) & (df_cem['consumo_fose'] <= 160)].copy()
ventana20['celda'] = ventana20['sector_cem'].astype(str) + '_' + ventana20['area'].astype(str) + '_' + ventana20['tamano_cat'].astype(str)

print(f"{'='*60}")
print(f"HETEROGENEIDAD POR SECTOR TÍPICO (CEM ±20 kWh)")
print(f"{'='*60}")

for sector in ['ST1-ST2', 'ST3-ST4', 'SER']:
    sub = ventana20[ventana20['sector_cem'] == sector].copy()

    celdas = sub.groupby('celda')['T'].agg(['sum', 'count'])
    celdas['n_control'] = celdas['count'] - celdas['sum']
    celdas = celdas[(celdas['sum'] > 0) & (celdas['n_control'] > 0)]

    sub_match = sub[sub['celda'].isin(celdas.index)]

    if len(celdas) == 0:
        print(f"\n{sector}: Sin celdas válidas")
        continue

    res = []
    for celda in celdas.index:
        c = sub_match[sub_match['celda'] == celda]
        t1 = c[c['T'] == 1]['sobrecarga_2m'].mean()
        t0 = c[c['T'] == 0]['sobrecarga_2m'].mean()
        n1 = (c['T'] == 1).sum()
        res.append({'att': t1 - t0, 'n1': n1})

    rdf = pd.DataFrame(res)
    rdf['peso'] = rdf['n1'] / rdf['n1'].sum()
    att = (rdf['att'] * rdf['peso']).sum()

    np.random.seed(42)
    att_boot = []
    for _ in range(2000):
        idx = np.random.choice(len(sub_match), size=len(sub_match), replace=True)
        boot = sub_match.iloc[idx]
        boot_res = []
        for celda in celdas.index:
            bc = boot[boot['celda'] == celda]
            t1b = bc[bc['T'] == 1]['sobrecarga_2m']
            t0b = bc[bc['T'] == 0]['sobrecarga_2m']
            if len(t1b) > 0 and len(t0b) > 0:
                boot_res.append({'att': t1b.mean() - t0b.mean(), 'n1': len(t1b)})
        if boot_res:
            bdf = pd.DataFrame(boot_res)
            bdf['peso'] = bdf['n1'] / bdf['n1'].sum()
            att_boot.append((bdf['att'] * bdf['peso']).sum())

    se = np.std(att_boot)
    pval = 2 * (1 - stats.norm.cdf(abs(att / se))) if se > 0 else 1

    print(f"\n{sector}: N={len(sub_match)}, T={sub_match['T'].sum()}, C={(sub_match['T']==0).sum()}, Celdas={len(celdas)}")
    print(f"  ATT = {att*100:.2f} pp, SE = {se*100:.2f}, p = {pval:.3f}")

# ============================================================
# HETEROGENEIDAD POR CONDICIÓN DE POBREZA (CEM ±20 kWh)
# ============================================================
# ¿El umbral discrimina distinto entre pobres y no pobres?
# Si el umbral funciona, debería separar más entre los pobres
# (donde la sobrecarga es más frecuente) que entre los no pobres.
# ============================================================

print(f"\n{'='*60}")
print(f"HETEROGENEIDAD POR CONDICIÓN DE POBREZA (CEM ±20 kWh)")
print(f"{'='*60}")

for label, mask in [('No pobre', ventana20['POBREZA'] == 'No pobre'),
                     ('Pobre no extremo', ventana20['POBREZA'] == 'Pobre no extremo'),
                     ('Pobre extremo', ventana20['POBREZA'] == 'Pobre extremo')]:
    sub = ventana20[mask].copy()

    celdas = sub.groupby('celda')['T'].agg(['sum', 'count'])
    celdas['n_control'] = celdas['count'] - celdas['sum']
    celdas = celdas[(celdas['sum'] > 0) & (celdas['n_control'] > 0)]

    sub_match = sub[sub['celda'].isin(celdas.index)]

    if len(celdas) == 0:
        print(f"\n{label}: Sin celdas válidas")
        continue

    res = []
    for celda in celdas.index:
        c = sub_match[sub_match['celda'] == celda]
        t1 = c[c['T'] == 1]['sobrecarga_2m'].mean()
        t0 = c[c['T'] == 0]['sobrecarga_2m'].mean()
        n1 = (c['T'] == 1).sum()
        res.append({'att': t1 - t0, 'n1': n1})

    rdf = pd.DataFrame(res)
    rdf['peso'] = rdf['n1'] / rdf['n1'].sum()
    att = (rdf['att'] * rdf['peso']).sum()

    np.random.seed(42)
    att_boot = []
    for _ in range(2000):
        idx = np.random.choice(len(sub_match), size=len(sub_match), replace=True)
        boot = sub_match.iloc[idx]
        boot_res = []
        for celda in celdas.index:
            bc = boot[boot['celda'] == celda]
            t1b = bc[bc['T'] == 1]['sobrecarga_2m']
            t0b = bc[bc['T'] == 0]['sobrecarga_2m']
            if len(t1b) > 0 and len(t0b) > 0:
                boot_res.append({'att': t1b.mean() - t0b.mean(), 'n1': len(t1b)})
        if boot_res:
            bdf = pd.DataFrame(boot_res)
            bdf['peso'] = bdf['n1'] / bdf['n1'].sum()
            att_boot.append((bdf['att'] * bdf['peso']).sum())

    se = np.std(att_boot)
    pval = 2 * (1 - stats.norm.cdf(abs(att / se))) if se > 0 else 1

    print(f"\n{label}: N={len(sub_match)}, T={sub_match['T'].sum()}, C={(sub_match['T']==0).sum()}")
    print(f"  ATT = {att*100:.2f} pp, SE = {se*100:.2f}, p = {pval:.3f}")

In [ ]:
# ============================================================
# SENSIBILIDAD POR ANCHO DE VENTANA (±10 a ±35 kWh)
# ============================================================
# ¿El resultado depende de la ventana elegida?
# Se estima el CEM para 6 ventanas simétricas alrededor de 140.
# Ventanas estrechas = más comparabilidad, menos observaciones.
# Ventanas amplias = más observaciones, más heterogeneidad.
# Si el ATT es estable → resultado robusto.
# Si solo es significativo en ventanas amplias → probablemente
# cambio composicional, no efecto del umbral.
# ============================================================

print(f"{'='*60}")
print(f"SENSIBILIDAD POR ANCHO DE VENTANA")
print(f"{'='*60}")
print(f"{'BW':<6} {'N':>6} {'T':>5} {'C':>5} {'Celdas':>7} {'ATT':>8} {'SE':>7} {'p':>7}")
print("-" * 55)

for bw in [10, 15, 20, 25, 30, 35]:
    low = 140 - bw
    high = 140 + bw
    v = df_cem[(df_cem['consumo_fose'] >= low) & (df_cem['consumo_fose'] <= high)].copy()
    v['celda'] = v['sector_cem'].astype(str) + '_' + v['area'].astype(str) + '_' + v['tamano_cat'].astype(str)

    celdas = v.groupby('celda')['T'].agg(['sum', 'count'])
    celdas['n_control'] = celdas['count'] - celdas['sum']
    celdas = celdas[(celdas['sum'] > 0) & (celdas['n_control'] > 0)]

    v_match = v[v['celda'].isin(celdas.index)]

    if len(celdas) == 0:
        print(f"±{bw:<4} Sin celdas válidas")
        continue

    res = []
    for celda in celdas.index:
        c = v_match[v_match['celda'] == celda]
        t1 = c[c['T'] == 1]['sobrecarga_2m'].mean()
        t0 = c[c['T'] == 0]['sobrecarga_2m'].mean()
        n1 = (c['T'] == 1).sum()
        res.append({'att': t1 - t0, 'n1': n1})

    rdf = pd.DataFrame(res)
    rdf['peso'] = rdf['n1'] / rdf['n1'].sum()
    att = (rdf['att'] * rdf['peso']).sum()

    np.random.seed(42)
    att_boot = []
    for _ in range(2000):
        idx = np.random.choice(len(v_match), size=len(v_match), replace=True)
        boot = v_match.iloc[idx]
        boot_res = []
        for celda in celdas.index:
            bc = boot[boot['celda'] == celda]
            t1b = bc[bc['T'] == 1]['sobrecarga_2m']
            t0b = bc[bc['T'] == 0]['sobrecarga_2m']
            if len(t1b) > 0 and len(t0b) > 0:
                boot_res.append({'att': t1b.mean() - t0b.mean(), 'n1': len(t1b)})
        if boot_res:
            bdf = pd.DataFrame(boot_res)
            bdf['peso'] = bdf['n1'] / bdf['n1'].sum()
            att_boot.append((bdf['att'] * bdf['peso']).sum())

    se = np.std(att_boot)
    pval = 2 * (1 - stats.norm.cdf(abs(att / se))) if se > 0 else 1
    nt = v_match['T'].sum()
    nc = (v_match['T'] == 0).sum()

    print(f"±{bw:<4} {len(v_match):>6} {nt:>5} {nc:>5} {len(celdas):>7} {att*100:>7.2f} {se*100:>7.2f} {pval:>7.3f}")

In [ ]:
# ============================================================
# BALANCE SMD ANTES Y DESPUÉS DEL CEM (±20 kWh)
# ============================================================
# La Diferencia Estandarizada de Medias (SMD) mide cuánto
# difieren tratados y controles en cada covariable.
# Regla: SMD < 0,10 después del matching = buen balance.
# Si el CEM funciona, todas las SMD deberían bajar a ~0.
# También incluimos variables NO usadas en el matching
# (N_PERSONAS continua) para verificar que el balance se
# extiende a variables no incluidas en la especificación.
# ============================================================

ventana20 = df_cem[(df_cem['consumo_fose'] >= 120) & (df_cem['consumo_fose'] <= 160)].copy()
ventana20['celda'] = ventana20['sector_cem'].astype(str) + '_' + ventana20['area'].astype(str) + '_' + ventana20['tamano_cat'].astype(str)

celdas = ventana20.groupby('celda')['T'].agg(['sum', 'count'])
celdas['n_control'] = celdas['count'] - celdas['sum']
celdas = celdas[(celdas['sum'] > 0) & (celdas['n_control'] > 0)]
ventana_match = ventana20[ventana20['celda'].isin(celdas.index)].copy()

# Crear dummies de las covariables
dummies = pd.get_dummies(ventana20[['sector_cem', 'area', 'tamano_cat']])
dummies_match = pd.get_dummies(ventana_match[['sector_cem', 'area', 'tamano_cat']])

def smd(t1, t0):
    d = t1.mean() - t0.mean()
    s = np.sqrt((t1.var() + t0.var()) / 2)
    return abs(d / s) if s > 0 else 0

print(f"{'='*60}")
print(f"BALANCE SMD ANTES Y DESPUÉS DEL CEM (±20 kWh)")
print(f"{'='*60}")
print(f"{'Variable':<25} {'SMD antes':>10} {'SMD después':>12}")
print("-" * 50)

for col in dummies.columns:
    t1_antes = dummies.loc[ventana20['T'] == 1, col]
    t0_antes = dummies.loc[ventana20['T'] == 0, col]
    smd_antes = smd(t1_antes, t0_antes)

    if col in dummies_match.columns:
        t1_desp = dummies_match.loc[ventana_match['T'] == 1, col]
        t0_desp = dummies_match.loc[ventana_match['T'] == 0, col]
        smd_desp = smd(t1_desp, t0_desp)
    else:
        smd_desp = 0

    print(f"{col:<25} {smd_antes:>10.4f} {smd_desp:>12.4f}")

# Tamaño del hogar continuo (no usada en el matching)
t1a = ventana20.loc[ventana20['T'] == 1, 'N_PERSONAS']
t0a = ventana20.loc[ventana20['T'] == 0, 'N_PERSONAS']
t1d = ventana_match.loc[ventana_match['T'] == 1, 'N_PERSONAS']
t0d = ventana_match.loc[ventana_match['T'] == 0, 'N_PERSONAS']
print(f"{'N_PERSONAS (continua)':<25} {smd(t1a, t0a):>10.4f} {smd(t1d, t0d):>12.4f}")

max_smd = max([smd(dummies_match.loc[ventana_match['T']==1, c], dummies_match.loc[ventana_match['T']==0, c]) for c in dummies_match.columns])
print(f"\nMáximo SMD después del CEM: {max_smd:.4f}")
print(f"¿Todas < 0,10? {'SÍ' if max_smd < 0.10 else 'NO'}")

# ============================================================
# OVERLAP: DISTRIBUCIÓN DEL PROPENSITY SCORE (±20 kWh)
# ============================================================
# Si las distribuciones del PS de tratados y controles se
# solapan completamente, no hay selección sobre observables
# dentro de la ventana. PS medias similares = grupos comparables.
# ============================================================

from sklearn.linear_model import LogisticRegression

X = pd.get_dummies(ventana_match[['sector_cem', 'area', 'tamano_cat']], drop_first=True).astype(float)
y = ventana_match['T'].values
lr = LogisticRegression(max_iter=1000)
lr.fit(X, y)
ventana_match['ps'] = lr.predict_proba(X)[:, 1]

print(f"\n{'='*60}")
print(f"PROPENSITY SCORE OVERLAP (±20 kWh)")
print(f"{'='*60}")
print(f"PS elegibles:  media={ventana_match.loc[ventana_match['T']==1, 'ps'].mean():.4f}, std={ventana_match.loc[ventana_match['T']==1, 'ps'].std():.4f}")
print(f"PS excluidos:  media={ventana_match.loc[ventana_match['T']==0, 'ps'].mean():.4f}, std={ventana_match.loc[ventana_match['T']==0, 'ps'].std():.4f}")
print(f"Soporte común: {'PERFECTO' if abs(ventana_match.loc[ventana_match['T']==1, 'ps'].mean() - ventana_match.loc[ventana_match['T']==0, 'ps'].mean()) < 0.05 else 'REVISAR'}")

In [ ]:
# ============================================================
# GRÁFICO 11: Distribución del consumo eléctrico mensual y umbral de elegibilidad FOSE
# ============================================================
# Histograma ponderado del consumo mensual (kWh) con el umbral
# de 140 kWh marcado. Bins de 5 kWh. Destaca el tramo 130-140
# con el ratio ponderado (0,81 = sin bunching neto).
# ============================================================

fig, ax = plt.subplots(figsize=(14, 6))

df_cons = df[df['consumo_fose'].notna() & (df['consumo_fose'] > 0)].copy()

# Bins de 5 kWh desde 55 hasta 225
bins = np.arange(55, 226, 5)

# Separar elegibles, tramo 130-140, y excluidos
eleg = df_cons[(df_cons['consumo_fose'] >= 55) & (df_cons['consumo_fose'] < 130)]
tramo = df_cons[(df_cons['consumo_fose'] >= 130) & (df_cons['consumo_fose'] < 140)]
excl = df_cons[(df_cons['consumo_fose'] >= 140) & (df_cons['consumo_fose'] <= 225)]

# Histograma ponderado
ax.hist(eleg['consumo_fose'], bins=bins, weights=eleg['Fact_exp_hog']/1000,
        color='#2B6E99', edgecolor='white', linewidth=0.5, label='Elegibles (≤140 kWh)')
ax.hist(tramo['consumo_fose'], bins=bins, weights=tramo['Fact_exp_hog']/1000,
        color='#C44E52', edgecolor='white', linewidth=0.5, label='Tramo [130-140) kWh')
ax.hist(excl['consumo_fose'], bins=bins, weights=excl['Fact_exp_hog']/1000,
        color='#808080', edgecolor='white', linewidth=0.5, label='Excluidos (>140 kWh)')

# Línea vertical del umbral
ax.axvline(x=140, color='black', linestyle='--', linewidth=2)
ax.text(141, ax.get_ylim()[1]*0.9, 'Umbral FOSE\n140 kWh', fontsize=11, fontweight='bold')

# Anotación del ratio
hog_130_140 = tramo['Fact_exp_hog'].sum() / 1000
ax.annotate(f'[130-140): {hog_130_140:.0f}k hog.\nRatio pond. = 0.81\n(sin bunching neto)',
            xy=(135, hog_130_140*0.5), fontsize=9, color='#C44E52',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor='#C44E52'))

ax.set_xlabel('Consumo eléctrico mensual (kWh)', fontsize=12)
ax.set_ylabel('Hogares representados (miles)', fontsize=12)
ax.legend(fontsize=10, loc='upper right')

fig.suptitle('Figura 1. Distribución del consumo eléctrico mensual y umbral de elegibilidad FOSE\n(ERCUE 2025; ponderado por factor de expansión; bins de 5 kWh)',
             fontsize=13, fontweight='bold', y=1.02)

plt.tight_layout()
plt.savefig('fig1_distribucion.png', dpi=300, bbox_inches='tight')
plt.show()
print("Guardada como fig1_distribucion.png")

In [ ]:
# ============================================================
# GRÁFICO 12: Energy burden y sobrecarga energética por decil de gasto y sector
# ============================================================
# Panel A: Energy burden promedio por decil de gasto
# Panel B: Tasa de sobrecarga 2M por sector típico
# Umbral correcto: 6,26%
# ============================================================

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5))

# --- Panel A: Energy burden por decil ---
df_eb = df[df['energy_burden'].notna() & (df['energy_burden'] > 0) & (df['energy_burden'] < 1)].copy()
df_eb['decil'] = pd.qcut(df_eb['ga_aj_total_pc'], 10, labels=range(1, 11))

eb_decil = df_eb.groupby('decil')['energy_burden'].mean() * 100

ax1.bar(range(1, 11), eb_decil.values, color='#2B6E99', width=0.7)
ax1.axhline(y=6.26, color='#C44E52', linestyle=':', linewidth=1.5, label='Umbral 2M (6,26%)')
ax1.set_xlabel('Decil de gasto per cápita\n(1=más pobre, 10=más rico)', fontsize=11)
ax1.set_ylabel('Energy burden promedio (%)', fontsize=11)
ax1.set_title('Panel A. Energy burden por decil de gasto', fontsize=12, fontweight='bold')
ax1.set_xticks(range(1, 11))
ax1.set_ylim(0, max(eb_decil.values) * 1.15)
ax1.legend(fontsize=9)

# --- Panel B: Tasa 2M por sector típico ---
df_st = df[df['sobrecarga_2m'].notna() & df['sector_tipico'].notna()].copy()
sectores = ['ST1', 'ST2', 'ST3', 'ST4', 'SER']
tasas = []
for st in sectores:
    sub = df_st[df_st['sector_tipico'] == st]
    tasas.append(sub['sobrecarga_2m'].mean() * 100)

colores = ['#2B6E99', '#4A8DB7', '#808080', '#808080', '#C44E52']
bars = ax2.bar(sectores, tasas, color=colores, width=0.6)

# Añadir etiquetas encima de cada barra
for bar, tasa in zip(bars, tasas):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
             f'{tasa:.1f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')

ax2.axhline(y=6.26, color='#C44E52', linestyle=':', linewidth=1.5)
ax2.set_xlabel('Sector típico', fontsize=11)
ax2.set_ylabel('Tasa de sobrecarga energética 2M (%)', fontsize=11)
ax2.set_title('Panel B. Tasa 2M por sector típico', fontsize=12, fontweight='bold')
ax2.set_ylim(0, max(tasas) * 1.2)

fig.suptitle('Figura 2. Energy burden y sobrecarga energética por decil de gasto y sector típico\n(ERCUE 2025; umbral 2M = 6,26%; ponderado por factor de expansión)',
             fontsize=13, fontweight='bold', y=1.02)

plt.tight_layout()
plt.savefig('fig2_energy_burden.png', dpi=300, bbox_inches='tight')
plt.show()
print("Guardada como fig2_energy_burden.png")

In [ ]:
# ============================================================
# GRÁFICO 10: Errores de focalización del FOSE por sector típico
# ============================================================
# Tres barras por sector: Error de Inclusión, Error de Exclusión,
# y Multipredio. Con los números corregidos:
# - Multipredio filtrado (sin espacios en blanco)
# - EE corregido (sin NaN de consumo)
# - Filtro estrato alto aplicado en ST1
# ============================================================

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(16, 5.5))

sectores = ['ST1', 'ST2', 'ST3', 'ST4', 'SER']
df_cc = df[df['consumo_fose'].notna() & df['sector_tipico'].notna()].copy()

ei_vals, ee_vals, multi_vals = [], [], []
for st in sectores:
    sub = df_cc[df_cc['sector_tipico'] == st]
    eleg = sub[sub['elegible'] == 1]
    pob = sub[sub['pobre'] == True]
    ei_vals.append((eleg['pobre'] == False).sum() / len(eleg) * 100 if len(eleg) > 0 else 0)
    ee_vals.append((pob['elegible'] == 0).sum() / len(pob) * 100 if len(pob) > 0 else 0)
    multi_vals.append(eleg['multipredio'].sum() / len(eleg) * 100 if len(eleg) > 0 else 0)

# Panel 1: Error de Inclusión
bars1 = ax1.bar(sectores, ei_vals, color='#E8713A', width=0.6)
for bar, val in zip(bars1, ei_vals):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             f'{val:.1f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax1.set_title('Error de Inclusión (EI)\n% elegibles no pobres', fontsize=11, fontweight='bold')
ax1.set_ylabel('EI (%)', fontsize=11)
ax1.set_ylim(0, 100)

# Panel 2: Error de Exclusión
bars2 = ax2.bar(sectores, ee_vals, color='#C44E52', width=0.6)
for bar, val in zip(bars2, ee_vals):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
             f'{val:.1f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax2.set_title('Error de Exclusión (EE)\n% pobres excluidos por consumo', fontsize=11, fontweight='bold')
ax2.set_ylabel('EE (%)', fontsize=11)
ax2.set_ylim(0, max(ee_vals) * 1.3)

# Panel 3: Multipredio
bars3 = ax3.bar(sectores, multi_vals, color='#2B6E99', width=0.6)
for bar, val in zip(bars3, multi_vals):
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
             f'{val:.1f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax3.set_title('Multipredio\n% elegibles con suministro compartido', fontsize=11, fontweight='bold')
ax3.set_ylabel('Multipredio (%)', fontsize=11)
ax3.set_ylim(0, max(multi_vals) * 1.3)

fig.suptitle('Figura 3. Errores de focalización del FOSE por sector típico (ERCUE 2025)',
             fontsize=13, fontweight='bold', y=1.02)

plt.tight_layout()
plt.savefig('fig3_focalizacion.png', dpi=300, bbox_inches='tight')
plt.show()
print("Guardada como fig3_focalizacion.png")

In [ ]:
# ============================================================
# GRÁFICO 13: Tasa de sobrecarga energética (indicador 2M) por tramo de consumo
# ============================================================
# Muestra cómo la tasa de sobrecarga varía con el consumo.
# La ventana ±20 kWh está sombreada para mostrar la zona
# donde se estima el CEM. El umbral 2M (6,26%) está marcado.
# Si la tasa es similar a ambos lados de 140 kWh → el umbral
# no separa niveles de sobrecarga distintos.
# ============================================================

fig, ax = plt.subplots(figsize=(14, 6))

df_t = df[df['consumo_fose'].notna() & df['sobrecarga_2m'].notna()].copy()

# Bins de 10 kWh
bins_edges = np.arange(60, 230, 10)
tasas = []
centros = []
colores = []

for i in range(len(bins_edges)-1):
    low, high = bins_edges[i], bins_edges[i+1]
    sub = df_t[(df_t['consumo_fose'] >= low) & (df_t['consumo_fose'] < high)]
    if len(sub) > 0:
        tasa = sub['sobrecarga_2m'].mean() * 100
        tasas.append(tasa)
        centros.append((low + high) / 2)
        # Color: azul si elegible, gris si excluido
        colores.append('#2B6E99' if high <= 140 else '#808080')

bars = ax.bar(centros, tasas, width=8, color=colores, edgecolor='white', linewidth=0.5)

# Umbral FOSE
ax.axvline(x=140, color='black', linestyle='--', linewidth=2)
ax.text(141, max(tasas)*0.95, 'Umbral FOSE\n140 kWh', fontsize=11, fontweight='bold')

# Umbral 2M
ax.axhline(y=6.26, color='#C44E52', linestyle=':', linewidth=1.5)
ax.text(62, 6.6, 'Umbral 2M (6,26%)', fontsize=9, color='#C44E52')

# Sombrear ventana ±20 kWh
ax.axvspan(120, 160, alpha=0.08, color='black', label='±20 kWh ventana')
ax.text(130, max(tasas)*0.1, '±20 kWh\nventana', fontsize=9, color='gray', ha='center')

ax.set_xlabel('Consumo eléctrico mensual (kWh)', fontsize=12)
ax.set_ylabel('Tasa de sobrecarga energética 2M (%)', fontsize=12)
ax.legend(['Elegibles (≤140 kWh)', 'Excluidos (>140 kWh)'], fontsize=10, loc='upper left')

fig.suptitle('Figura 4. Tasa de sobrecarga energética (indicador 2M) por tramo de consumo\n(ERCUE 2025; medias ponderadas; bins de 10 kWh)',
             fontsize=13, fontweight='bold', y=1.02)

plt.tight_layout()
plt.savefig('fig4_sobrecarga_consumo.png', dpi=300, bbox_inches='tight')
plt.show()
print("Guardada como fig4_sobrecarga_consumo.png")

In [ ]:
# ============================================================
# GRÁFICO 14: Diferencia en sobrecarga energética (2M) asociada a la elegibilidad FOSE: Comparación de estimadores
# ============================================================
# Muestra el ATT de cada estimador con sus intervalos de
# confianza al 90% (grueso) y 95% (fino). La línea vertical
# en cero = resultado nulo. Si el IC cruza el cero, no hay
# diferencia significativa.
# ============================================================

fig, ax = plt.subplots(figsize=(12, 6))

# Resultados definitivos (con filtro estrato alto)
estimadores = [
    ('CEM ±20 kWh\n(principal)', -2.37, 2.43, 0.330),
    ('CEM ±30 kWh\n(robustez)', -5.72, 1.94, 0.003),
    ('IPW ±20 kWh\n(robustez)', -2.09, 2.35, 0.376),
    ('Sin matching\n±20 kWh', -1.03, None, None),
]

y_pos = [3, 2, 1, 0]

for i, (label, att, se, pval) in enumerate(estimadores):
    y = y_pos[i]

    # Color según significancia
    if pval is not None and pval < 0.10:
        color = '#C44E52'
        marker = 's'
    else:
        color = '#2B6E99'
        marker = 'o'

    ax.plot(att, y, marker=marker, markersize=12, color=color, zorder=5)

    if se is not None:
        # IC 95% (fino)
        ci95_low = att - 1.96 * se
        ci95_high = att + 1.96 * se
        ax.plot([ci95_low, ci95_high], [y, y], color=color, linewidth=1.5, alpha=0.4)

        # IC 90% (grueso)
        ci90_low = att - 1.645 * se
        ci90_high = att + 1.645 * se
        ax.plot([ci90_low, ci90_high], [y, y], color=color, linewidth=4, alpha=0.5)

        # p-valor
        sig = 'n.s.' if pval >= 0.10 else f'p={pval:.3f}'
        ax.text(ci95_high + 0.3, y, sig, fontsize=10, va='center', color=color)
    else:
        ax.plot([att-0.5, att+0.5], [y, y], color='gray', linewidth=1, linestyle=':')

# Línea de cero
ax.axvline(x=0, color='black', linewidth=1)

# Zona de resultado nulo
ax.axvspan(-2, 2, alpha=0.05, color='gray')
ax.text(0, 3.6, 'Zona de resultado nulo', fontsize=9, color='gray', ha='center', style='italic')

ax.set_yticks(y_pos)
ax.set_yticklabels([e[0] for e in estimadores], fontsize=11)
ax.set_xlabel('Efecto estimado sobre la tasa de sobrecarga energética (2M)\n(puntos porcentuales; IC 90% grueso, IC 95% fino)', fontsize=11)

# Leyenda
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#2B6E99', markersize=10, label='p ≥ 0,10'),
    Line2D([0], [0], marker='s', color='w', markerfacecolor='#C44E52', markersize=10, label='p < 0,10'),
    Line2D([0], [0], color='#2B6E99', linewidth=4, alpha=0.5, label='IC 90%'),
    Line2D([0], [0], color='#2B6E99', linewidth=1.5, alpha=0.4, label='IC 95%'),
]
ax.legend(handles=legend_elements, loc='lower left', fontsize=9)

fig.suptitle('Figura 5. Diferencias en sobrecarga energética (indicador 2M)\nasociadas a la elegibilidad FOSE — Comparación de estimadores',
             fontsize=13, fontweight='bold', y=1.02)

plt.tight_layout()
plt.savefig('fig5_forest_plot.png', dpi=300, bbox_inches='tight')
plt.show()
print("Guardada como fig5_forest_plot.png")

In [ ]:
# ============================================================
# GRÁFICO 17: Sensibilidad del ATT al ancho de ventana (CEM)
# ============================================================
# ¿El resultado depende de la ventana elegida?
# Ventanas estrechas (±10 a ±20): máxima comparabilidad → ATT ~0
# Ventanas amplias (±30 a ±35): más heterogeneidad → ATT crece
# Si el ATT solo es significativo en ventanas amplias, la
# diferencia refleja cambio composicional, no efecto del umbral.
# ============================================================

fig, ax = plt.subplots(figsize=(12, 6))

# Resultados definitivos (con filtro estrato alto)
bws =    [10,    15,    20,    25,    30,    35]
atts =   [-5.85, -4.61, -2.37, -3.89, -5.72, -6.40]
ses =    [3.51,  2.86,  2.43,  2.02,  1.94,  1.86]
pvals =  [0.096, 0.107, 0.330, 0.054, 0.003, 0.001]
ns =     [578,   920,   1284,  1648,  1945,  2227]

x = range(len(bws))
labels = [f'±{b}' for b in bws]

for i in range(len(bws)):
    att = atts[i]
    se = ses[i]
    pval = pvals[i]

    color = '#C44E52' if pval < 0.10 else '#2B6E99'
    marker = 's' if pval < 0.10 else 'o'

    # IC 95% (fino)
    ci95_low = att - 1.96 * se
    ci95_high = att + 1.96 * se
    ax.plot([i, i], [ci95_low, ci95_high], color='#AAAAAA', linewidth=1.5)

    # IC 90% (grueso)
    ci90_low = att - 1.645 * se
    ci90_high = att + 1.645 * se
    ax.plot([i, i], [ci90_low, ci90_high], color=color, linewidth=4, alpha=0.6)

    # Punto
    ax.plot(i, att, marker=marker, markersize=12, color=color, zorder=5)

    # N y p-valor encima
    ax.text(i, ci95_high + 0.5, f'N={ns[i]}, p={pval:.3f}',
            ha='center', fontsize=8, color='gray')

# Línea de cero
ax.axhline(y=0, color='black', linestyle='--', linewidth=1)

ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=12)
ax.set_xlabel('Semiancho de ventana (kWh)', fontsize=12)
ax.set_ylabel('ATT (puntos porcentuales)', fontsize=12)

# Leyenda
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#2B6E99', markersize=10, label='p ≥ 0,10'),
    Line2D([0], [0], marker='s', color='w', markerfacecolor='#C44E52', markersize=10, label='p < 0,10'),
    Line2D([0], [0], color='#2B6E99', linewidth=4, alpha=0.6, label='IC 90%'),
    Line2D([0], [0], color='#AAAAAA', linewidth=1.5, label='IC 95%'),
]
ax.legend(handles=legend_elements, loc='lower left', fontsize=9)

fig.suptitle('Figura 6. Robustez por ancho de ventana: diferencia en sobrecarga energética (2M)\nCEM; covariables: sector típico, área, tamaño del hogar; IC 90% (oscuro) e IC 95% (claro)',
             fontsize=12, fontweight='bold', y=1.02)

plt.tight_layout()
plt.savefig('fig6_sensibilidad.png', dpi=300, bbox_inches='tight')
plt.show()
print("Guardada como fig6_sensibilidad.png")

In [ ]:
# ============================================================
# GRÁFICO 15: Balance de covariables antes y después del CEM
# ============================================================
# Muestra la Diferencia Estandarizada de Medias (SMD) para
# cada covariable. Línea punteada en 0,10 = umbral convencional.
# Si todas las SMD están por debajo después del CEM, el
# emparejamiento logra balance adecuado.
# ============================================================

fig, ax = plt.subplots(figsize=(10, 7))

# Resultados del balance (números verificados)
variables = [
    'Sector ST3-ST4',
    'Hogar 5-6 pers.',
    'Sector ST1-ST2',
    'Hogar 7+ pers.',
    'Sector SER',
    'Área rural',
    'Área urbana',
    'Hogar 3-4 pers.',
    'Hogar 1-2 pers.',
    'Tamaño hogar (cont.)',
]

smd_antes = [0.0446, 0.0079, 0.0095, 0.0307, 0.0569, 0.0341, 0.0341, 0.0763, 0.0829, 0.0947]
smd_despues = [0.0447, 0.0081, 0.0138, 0.0230, 0.0495, 0.0479, 0.0479, 0.0796, 0.0833, 0.0900]

y_pos = range(len(variables))

# Puntos antes y después con líneas conectoras
for i in range(len(variables)):
    ax.plot([smd_antes[i], smd_despues[i]], [i, i], color='gray', linewidth=1, zorder=1)

ax.scatter(smd_antes, y_pos, color='#C44E52', s=80, zorder=3, label='Antes del CEM')
ax.scatter(smd_despues, y_pos, color='#2B6E99', s=100, marker='D', zorder=3, label='Después del CEM')

# Umbral 0,10
ax.axvline(x=0.10, color='gray', linestyle='--', linewidth=1.5, label='Umbral |SMD| = 0,10')

ax.set_yticks(y_pos)
ax.set_yticklabels(variables, fontsize=11)
ax.set_xlabel('|SMD|', fontsize=12)
ax.set_xlim(-0.005, 0.12)
ax.legend(fontsize=10, loc='lower right')

fig.suptitle('Figura 7. Balance de covariables antes y después del CEM\n(ventana ±20 kWh; línea punteada = umbral |SMD| = 0,10)',
             fontsize=13, fontweight='bold', y=1.02)

plt.tight_layout()
plt.savefig('fig7_balance_smd.png', dpi=300, bbox_inches='tight')
plt.show()
print("Guardada como fig7_balance_smd.png")

In [ ]:
# ============================================================
# GRÁFICO 16: Distribución del propensity score por grupo
# ============================================================
# Si las dos distribuciones se solapan completamente, no hay
# selección sobre observables dentro de la ventana.
# PS medias casi idénticas (0,575 vs 0,571) = soporte perfecto.
# ============================================================

fig, ax = plt.subplots(figsize=(10, 6))

# Recalcular PS para la ventana ±20 kWh
ventana20 = df_cem[(df_cem['consumo_fose'] >= 120) & (df_cem['consumo_fose'] <= 160)].copy()
ventana20['celda'] = ventana20['sector_cem'].astype(str) + '_' + ventana20['area'].astype(str) + '_' + ventana20['tamano_cat'].astype(str)

celdas = ventana20.groupby('celda')['T'].agg(['sum', 'count'])
celdas['n_control'] = celdas['count'] - celdas['sum']
celdas = celdas[(celdas['sum'] > 0) & (celdas['n_control'] > 0)]
v_match = ventana20[ventana20['celda'].isin(celdas.index)].copy()

from sklearn.linear_model import LogisticRegression
X = pd.get_dummies(v_match[['sector_cem', 'area', 'tamano_cat']], drop_first=True).astype(float)
y = v_match['T'].values
lr = LogisticRegression(max_iter=1000)
lr.fit(X, y)
v_match['ps'] = lr.predict_proba(X)[:, 1]

ps_eleg = v_match.loc[v_match['T'] == 1, 'ps']
ps_excl = v_match.loc[v_match['T'] == 0, 'ps']

# Histogramas superpuestos
bins = np.linspace(0, 1, 50)
ax.hist(ps_eleg, bins=bins, alpha=0.5, color='#2B6E99', density=True, label=f'Elegibles (≤140 kWh)')
ax.hist(ps_excl, bins=bins, alpha=0.5, color='#C44E52', density=True, label=f'Excluidos (>140 kWh)')

# Medias
ax.axvline(x=ps_eleg.mean(), color='#2B6E99', linestyle='--', linewidth=2, label=f'Media elegibles: {ps_eleg.mean():.3f}')
ax.axvline(x=ps_excl.mean(), color='#C44E52', linestyle='-.', linewidth=2, label=f'Media excluidos: {ps_excl.mean():.3f}')

ax.set_xlabel('Propensity Score estimado', fontsize=12)
ax.set_ylabel('Densidad', fontsize=12)
ax.legend(fontsize=10)

fig.suptitle('Figura 8. Distribución del propensity score por grupo\n(ventana ±20 kWh; logit con sector típico, área y tamaño del hogar)',
             fontsize=13, fontweight='bold', y=1.02)

plt.tight_layout()
plt.savefig('fig8_overlap.png', dpi=300, bbox_inches='tight')
plt.show()
print("Guardada como fig8_overlap.png")

In [ ]:
import os, glob
# Buscar el archivo
for f in glob.glob('/*Evaluaci*') + glob.glob('/*FOSE*') + glob.glob('/*MCTER*'):
    print(f)
for f in os.listdir('.'):
    if 'valuaci' in f.lower() or 'fose' in f.lower() or 'mcter' in f.lower():
        print(f)

In [ ]:
# ============================================================
# GRÁFICO 20: Simulación contrafactual: energy burden con y sin FOSE
# ============================================================
import numpy as np

fose_excel = pd.read_excel('Evaluación FOSE y MCTER nueva metodología  vf 0606 vf (1).xlsx', sheet_name='FOSE', header=7)

tarifas_cf = fose_excel[['Código Sistema', 'De 0 a 30 kw', 'De 30 a 140 kw', 'Más de 140 kw']].copy()
tarifas_cf.columns = ['sistema', 'tarifa_cf_b1', 'tarifa_cf_b2', 'tarifa_cf_b3']
tarifas_cf['sistema'] = tarifas_cf['sistema'].str.strip()

tarifas_sf = fose_excel[['Código Sistema', 'De 0 a 30 kw.2', 'De 30 a 140 kw.2', 'Más de 140 kw.2']].copy()
tarifas_sf.columns = ['sistema', 'tarifa_sf_b1', 'tarifa_sf_b2', 'tarifa_sf_b3']
tarifas_sf['sistema'] = tarifas_sf['sistema'].str.strip()

sim = df[df['consumo_fose'].notna() & df['gasto_mes_electricidad'].notna() &
         df['sector_tipico'].notna() & (df['consumo_fose'] > 0) & (df['elegible'] == 1)].copy()
sim['sistema'] = sim['sistema'].str.strip()
sim = sim.merge(tarifas_cf, on='sistema', how='left')
sim = sim.merge(tarifas_sf, on='sistema', how='left')

b1 = sim['consumo_fose'].clip(upper=30)
b2 = (sim['consumo_fose'] - 30).clip(lower=0, upper=110)
b3 = (sim['consumo_fose'] - 140).clip(lower=0)

sim['fac_con'] = b1 * sim['tarifa_cf_b1'] + b2 * sim['tarifa_cf_b2'] + b3 * sim['tarifa_cf_b3']
sim['fac_sin'] = b1 * sim['tarifa_sf_b1'] + b2 * sim['tarifa_sf_b2'] + b3 * sim['tarifa_sf_b3']
sim['descuento'] = sim['fac_sin'] - sim['fac_con']
sim['gasto_sin_fose'] = sim['gasto_mes_electricidad'] + sim['descuento']

gasto_total = sim['ga_aj_total_pc'] * sim['N_PERSONAS']
sim['eb_con'] = sim['gasto_mes_electricidad'] / gasto_total
sim['eb_sin'] = sim['gasto_sin_fose'] / gasto_total

umbral = 0.0626

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5))

sim_valid = sim[sim['eb_con'].notna() & sim['eb_sin'].notna() &
                (sim['eb_con'] < 0.5) & (sim['eb_sin'] < 0.5)]

ax1.hist(sim_valid['eb_con']*100, bins=50, alpha=0.5, color='#2B6E99', density=True, label='Con FOSE')
ax1.hist(sim_valid['eb_sin']*100, bins=50, alpha=0.5, color='#C44E52', density=True, label='Sin FOSE')
ax1.axvline(x=6.26, color='black', linestyle='--', linewidth=1.5, label='Umbral 2M (6,26%)')
ax1.set_xlabel('Energy burden (%)', fontsize=11)
ax1.set_ylabel('Densidad', fontsize=11)
ax1.set_title('A. Distribución del energy burden\n(hogares elegibles, ≤140 kWh)', fontsize=11, fontweight='bold')
ax1.legend(fontsize=9)

sectores = ['ST1', 'ST2', 'ST3', 'ST4', 'SER']
con_vals, sin_vals = [], []
for st in sectores:
    sub = sim[sim['sector_tipico'] == st]
    con_vals.append((sub['eb_con'] > umbral).mean() * 100)
    sin_vals.append((sub['eb_sin'] > umbral).mean() * 100)

x = np.arange(len(sectores))
w = 0.35

bars1 = ax2.bar(x - w/2, con_vals, w, color='#2B6E99', label='Con FOSE')
bars2 = ax2.bar(x + w/2, sin_vals, w, color='#C44E52', label='Sin FOSE')

for bar, val in zip(bars1, con_vals):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
             f'{val:.1f}%', ha='center', fontsize=9, color='#2B6E99')
for bar, val in zip(bars2, sin_vals):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
             f'{val:.1f}%', ha='center', fontsize=9, color='#C44E52')

ax2.set_xticks(x)
ax2.set_xticklabels(sectores, fontsize=11)
ax2.set_xlabel('Sector típico', fontsize=11)
ax2.set_ylabel('Tasa de sobrecarga energética 2M (%)', fontsize=11)
ax2.set_title('B. Tasa de sobrecarga energética (2M)\npor sector típico', fontsize=11, fontweight='bold')
ax2.legend(fontsize=10)

fig.suptitle('Figura 9. Simulación contrafactual: energy burden e incidencia de sobrecarga con y sin FOSE\n(ERCUE 2025; hogares elegibles ≤140 kWh con tarifa disponible)',
             fontsize=12, fontweight='bold', y=1.02)

plt.tight_layout()
plt.savefig('fig9_simulacion.png', dpi=300, bbox_inches='tight')
plt.show()
print("Guardada como fig9_simulacion.png")

# Tabla resumen
print(f"\n{'Sector':<8} {'N':>6} {'Con FOSE':>9} {'Sin FOSE':>9} {'Δ pp':>7} {'Desc. med.':>11}")
print("-" * 52)
for st in sectores:
    sub = sim[sim['sector_tipico'] == st]
    con = (sub['eb_con'] > umbral).mean() * 100
    sin = (sub['eb_sin'] > umbral).mean() * 100
    desc = sub['descuento'].median()
    print(f"{st:<8} {len(sub):>6} {con:>8.1f}% {sin:>8.1f}% {sin-con:>+6.1f} S/{desc:>9.2f}")

In [ ]:
# ============================================================
# GRÁFICO 18: Heterogeneidad por condición de pobreza (CEM ±20 kWh)
# ============================================================
# Resultado EXPLORATORIO por submuestras pequeñas.
# El umbral parece separar más entre pobres no extremos
# (donde la sobrecarga es más frecuente) que entre no pobres.
# Pobre extremo no es estimable (N=9, sin varianza).
# ============================================================

fig, ax = plt.subplots(figsize=(10, 6))

categorias = ['No pobre\n(N=1.132)', 'Pobre no extremo\n(N=121)', 'Pobre extremo\n(N=9)']
atts = [-0.23, -20.29, None]  # None para no estimable
ses = [2.19, 8.22, None]
pvals = [0.918, 0.014, None]

y_pos = [2, 1, 0]

for i in range(len(categorias)):
    att = atts[i]
    se = ses[i]
    pval = pvals[i]
    y = y_pos[i]

    if att is None:
        # No estimable
        ax.plot(0, y, marker='x', markersize=14, color='gray', zorder=5)
        ax.text(1, y, 'No estimable\n(N=9, sin varianza)', fontsize=10, va='center', color='gray', style='italic')
        continue

    color = '#C44E52' if pval < 0.05 else '#2B6E99'
    marker = 's' if pval < 0.05 else 'o'

    ax.plot(att, y, marker=marker, markersize=14, color=color, zorder=5)

    ci95_low = att - 1.96 * se
    ci95_high = att + 1.96 * se
    ci90_low = att - 1.645 * se
    ci90_high = att + 1.645 * se

    ax.plot([ci95_low, ci95_high], [y, y], color=color, linewidth=1.5, alpha=0.4)
    ax.plot([ci90_low, ci90_high], [y, y], color=color, linewidth=4, alpha=0.5)

    sig = f'p={pval:.3f}' if pval < 0.05 else 'n.s.'
    ax.text(ci95_high + 0.5, y, sig, fontsize=11, va='center', color=color, fontweight='bold')

ax.axvline(x=0, color='black', linewidth=1)
ax.axvspan(-3, 3, alpha=0.05, color='gray')

ax.set_yticks(y_pos)
ax.set_yticklabels(categorias, fontsize=12)
ax.set_xlabel('ATT: diferencia en tasa de sobrecarga 2M (puntos porcentuales)\nNegativo = excluidos tienen más sobrecarga que elegibles', fontsize=11)

ax.annotate('Brecha de 20 pp entre pobres\nelegibles y excluidos\n(evidencia exploratoria)',
            xy=(-20.29, 1), xytext=(-35, 1.6),
            fontsize=10, color='#C44E52',
            arrowprops=dict(arrowstyle='->', color='#C44E52', linewidth=1.5))

from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker='s', color='w', markerfacecolor='#C44E52', markersize=10, label='p < 0,05'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#2B6E99', markersize=10, label='No significativo'),
    Line2D([0], [0], marker='x', color='w', markerfacecolor='gray', markersize=10, label='No estimable'),
]
ax.legend(handles=legend_elements, loc='lower left', fontsize=10)

fig.suptitle('Figura 10. Heterogeneidad por condición de pobreza (CEM ±20 kWh)\nEvidencia exploratoria: submuestras limitadas, inferencia cauta',
             fontsize=13, fontweight='bold', y=1.02)

plt.tight_layout()
plt.savefig('fig10_heterogeneidad_pobreza.png', dpi=300, bbox_inches='tight')
plt.show()
print("Guardada como fig10_heterogeneidad_pobreza.png")

In [ ]:
# ============================================================
# SENSIBILIDAD: ¿El −20 pp en pobres no extremos es robusto?
# ============================================================
# Si el efecto se mantiene en distintas ventanas → hallazgo real
# Si solo aparece en ±20 y desaparece en otras → frágil
# ============================================================

print(f"{'='*60}")
print(f"SENSIBILIDAD POBRES NO EXTREMOS POR BANDWIDTH")
print(f"{'='*60}")
print(f"{'BW':<6} {'N':>6} {'T':>5} {'C':>5} {'ATT':>8} {'SE':>7} {'p':>7}")
print("-" * 48)

for bw in [15, 20, 25, 30]:
    low = 140 - bw
    high = 140 + bw
    v = df_cem[(df_cem['consumo_fose'] >= low) & (df_cem['consumo_fose'] <= high)].copy()
    v = v[v['POBREZA'] == 'Pobre no extremo'].copy()
    v['celda'] = v['sector_cem'].astype(str) + '_' + v['area'].astype(str) + '_' + v['tamano_cat'].astype(str)

    celdas = v.groupby('celda')['T'].agg(['sum', 'count'])
    celdas['n_control'] = celdas['count'] - celdas['sum']
    celdas = celdas[(celdas['sum'] > 0) & (celdas['n_control'] > 0)]

    v_match = v[v['celda'].isin(celdas.index)]

    if len(celdas) == 0:
        print(f"±{bw:<4} Sin celdas válidas")
        continue

    res = []
    for celda in celdas.index:
        c = v_match[v_match['celda'] == celda]
        t1 = c[c['T'] == 1]['sobrecarga_2m'].mean()
        t0 = c[c['T'] == 0]['sobrecarga_2m'].mean()
        n1 = (c['T'] == 1).sum()
        res.append({'att': t1 - t0, 'n1': n1})

    rdf = pd.DataFrame(res)
    rdf['peso'] = rdf['n1'] / rdf['n1'].sum()
    att = (rdf['att'] * rdf['peso']).sum()

    np.random.seed(42)
    att_boot = []
    for _ in range(2000):
        idx = np.random.choice(len(v_match), size=len(v_match), replace=True)
        boot = v_match.iloc[idx]
        boot_res = []
        for celda in celdas.index:
            bc = boot[boot['celda'] == celda]
            t1b = bc[bc['T'] == 1]['sobrecarga_2m']
            t0b = bc[bc['T'] == 0]['sobrecarga_2m']
            if len(t1b) > 0 and len(t0b) > 0:
                boot_res.append({'att': t1b.mean() - t0b.mean(), 'n1': len(t1b)})
        if boot_res:
            bdf = pd.DataFrame(boot_res)
            bdf['peso'] = bdf['n1'] / bdf['n1'].sum()
            att_boot.append((bdf['att'] * bdf['peso']).sum())

    se = np.std(att_boot)
    pval = 2 * (1 - stats.norm.cdf(abs(att / se))) if se > 0 else 1
    nt = v_match['T'].sum()
    nc = (v_match['T'] == 0).sum()

    print(f"±{bw:<4} {len(v_match):>6} {nt:>5} {nc:>5} {att*100:>7.2f} {se*100:>7.2f} {pval:>7.3f}")

In [ ]:
import pandas as pd
df = pd.read_parquet('base_ercue2025_st.parquet')

In [ ]:
# ============================================================
# CUADRO 10: Indicadores de focalización según definición de elegibilidad (ponderados y no ponderados)
# ============================================================
# Universo:
#   - consumo_fose observado (notna)
#   - sector_tipico no faltante (notna)
#   - elegible no NaN (para no tratar faltantes como excluidos)
#
# Pesos:
#   - Fact_exp_hog (factor de expansión del hogar)
#
# Métricas:
#   - EI (Error de Inclusión): % elegibles que NO son pobres
#   - EE (Error de Exclusión): % pobres que NO son elegibles (elegible==0)
#   - Multi (Multipredio): % elegibles con multipredio==1
#
# Salidas:
#   1) Global: EI/EE/Multi ponderado y sin ponderar
#   2) Por sector_tipico: EI/EE/Multi ponderado y sin ponderar
#   3) Tabla final (DataFrame) y export opcional
# ============================================================

import numpy as np
import pandas as pd

# -----------------------------
# 0) Parámetros editables
# -----------------------------
SECTORES = ["ST1", "ST2", "ST3", "ST4", "SER"]
WCOL = "Fact_exp_hog"

# -----------------------------
# 1) Filtrado de universo
# -----------------------------
df_cc = df[df["consumo_fose"].notna() & df["sector_tipico"].notna()].copy()

# Limpiar pesos
df_cc["w"] = pd.to_numeric(df_cc[WCOL], errors="coerce").fillna(0.0)

# Asegurar 0/1 en pobres, multipredio y elegible
df_cc["pobre01"] = df_cc["pobre"].fillna(False).astype(int)
df_cc["multipredio01"] = df_cc["multipredio"].fillna(False).astype(int)

# elegible puede tener NaN si consumo_fose es NaN (pero ya filtramos consumo),
# aun así, protegemos por si quedó NaN por otras reglas
df_cc["elegible01"] = pd.to_numeric(df_cc["elegible"], errors="coerce")
df_cc = df_cc[df_cc["elegible01"].notna()].copy()
df_cc["elegible01"] = df_cc["elegible01"].astype(int)

# (Opcional) descartar pesos <= 0 por seguridad (si sabes que nunca deberían existir)
# df_cc = df_cc[df_cc["w"] > 0].copy()

# -----------------------------
# 2) Funciones auxiliares
# -----------------------------
def share_weighted(mask_num, mask_den, w):
    """Porcentaje ponderado: sum(w * 1[num]) / sum(w * 1[den]) * 100"""
    den = w[mask_den].sum()
    if den <= 0:
        return np.nan
    return (w[mask_num].sum() / den) * 100

def share_unweighted(mask_num, mask_den):
    """Porcentaje muestral: sum(1[num]) / sum(1[den]) * 100"""
    den = mask_den.sum()
    if den <= 0:
        return np.nan
    return (mask_num.sum() / den) * 100

def compute_metrics(sub):
    """Devuelve EI/EE/Multi ponderado y no ponderado para un subset."""
    w = sub["w"]

    eleg = (sub["elegible01"] == 1)
    pob  = (sub["pobre01"] == 1)

    # EI: elegibles que NO son pobres
    ei_w = share_weighted(mask_num=(eleg & (sub["pobre01"] == 0)), mask_den=eleg, w=w)
    ei_u = share_unweighted(mask_num=(eleg & (sub["pobre01"] == 0)), mask_den=eleg)

    # EE: pobres que NO son elegibles
    ee_w = share_weighted(mask_num=(pob & (sub["elegible01"] == 0)), mask_den=pob, w=w)
    ee_u = share_unweighted(mask_num=(pob & (sub["elegible01"] == 0)), mask_den=pob)

    # Multipredio: elegibles con multipredio
    mu_w = share_weighted(mask_num=(eleg & (sub["multipredio01"] == 1)), mask_den=eleg, w=w)
    mu_u = share_unweighted(mask_num=(eleg & (sub["multipredio01"] == 1)), mask_den=eleg)

    # Counts útiles
    N = len(sub)
    N_eleg = int(eleg.sum())
    N_pob  = int(pob.sum())
    W = float(w.sum())
    W_eleg = float(w[eleg].sum())
    W_pob  = float(w[pob].sum())

    return {
        "N_muestra": N,
        "N_eleg_muestra": N_eleg,
        "N_pob_muestra": N_pob,
        "W_poblacion": W,
        "W_eleg_poblacion": W_eleg,
        "W_pob_poblacion": W_pob,
        "EI_no_pond": ei_u,
        "EI_pond": ei_w,
        "EE_no_pond": ee_u,
        "EE_pond": ee_w,
        "Multi_no_pond": mu_u,
        "Multi_pond": mu_w
    }

# -----------------------------
# 3) Resultados globales
# -----------------------------
global_res = compute_metrics(df_cc)

print("=== UNIVERSO (ANEXO) ===")
print(f"N (muestra): {global_res['N_muestra']:,}")
print(f"Suma pesos (población): {global_res['W_poblacion']:,.0f} hogares")

print("\n=== ERRORES GLOBALES ===")
print(f"EI sin ponderar: {global_res['EI_no_pond']:.1f}%")
print(f"EI ponderado   : {global_res['EI_pond']:.1f}%")
print(f"EE sin ponderar: {global_res['EE_no_pond']:.1f}%")
print(f"EE ponderado   : {global_res['EE_pond']:.1f}%")
print(f"Multi sin pond.: {global_res['Multi_no_pond']:.1f}%")
print(f"Multi ponderado: {global_res['Multi_pond']:.1f}%")

# -----------------------------
# 4) Resultados por sector típico
# -----------------------------
rows = []
for st in SECTORES:
    sub = df_cc[df_cc["sector_tipico"] == st].copy()
    if len(sub) == 0:
        rows.append({
            "Sector": st,
            "N_muestra": 0,
            "W_poblacion": 0.0,
            "EI_no_pond": np.nan, "EI_pond": np.nan,
            "EE_no_pond": np.nan, "EE_pond": np.nan,
            "Multi_no_pond": np.nan, "Multi_pond": np.nan
        })
        continue

    r = compute_metrics(sub)
    rows.append({
        "Sector": st,
        "N_muestra": r["N_muestra"],
        "W_poblacion": r["W_poblacion"],
        "EI_no_pond": r["EI_no_pond"], "EI_pond": r["EI_pond"],
        "EE_no_pond": r["EE_no_pond"], "EE_pond": r["EE_pond"],
        "Multi_no_pond": r["Multi_no_pond"], "Multi_pond": r["Multi_pond"]
    })

df_sector = pd.DataFrame(rows)

# Añadir fila global al final
df_global_row = pd.DataFrame([{
    "Sector": "GLOBAL",
    "N_muestra": global_res["N_muestra"],
    "W_poblacion": global_res["W_poblacion"],
    "EI_no_pond": global_res["EI_no_pond"], "EI_pond": global_res["EI_pond"],
    "EE_no_pond": global_res["EE_no_pond"], "EE_pond": global_res["EE_pond"],
    "Multi_no_pond": global_res["Multi_no_pond"], "Multi_pond": global_res["Multi_pond"]
}])

df_out = pd.concat([df_sector, df_global_row], ignore_index=True)

# -----------------------------
# 5) Impresión bonita
# -----------------------------
print("\n=== ERRORES POR SECTOR TÍPICO (NO PONDERADO vs PONDERADO) ===")
print(f"{'Sector':<8} {'N':>6} {'W':>10} | {'EI%':>6} {'EIw%':>6} | {'EE%':>6} {'EEw%':>6} | {'M%':>6} {'Mw%':>6}")
print("-" * 78)

for _, r in df_out.iterrows():
    sec = r["Sector"]
    N   = int(r["N_muestra"])
    W   = r["W_poblacion"]
    ei  = r["EI_no_pond"]; eiw = r["EI_pond"]
    ee  = r["EE_no_pond"]; eew = r["EE_pond"]
    mu  = r["Multi_no_pond"]; muw = r["Multi_pond"]

    def fmt(x):
        return " n/a " if pd.isna(x) else f"{x:5.1f}"

    print(f"{sec:<8} {N:>6} {W:>10.0f} | {fmt(ei):>6} {fmt(eiw):>6} | {fmt(ee):>6} {fmt(eew):>6} | {fmt(mu):>6} {fmt(muw):>6}")

# -----------------------------
# 6) Export opcional
# -----------------------------
# df_out.to_csv("anexo_errores_focalizacion_ponderado_vs_no.csv", index=False)
# print("\nGuardado: anexo_errores_focalizacion_ponderado_vs_no.csv")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from google.colab import files

df = pd.read_parquet('base_ercue2025_st.parquet')
print(f"Cargado: {df.shape[0]} hogares")

# ============================================================
# GRÁFICO 11 (versión alternativa): Distribución del consumo eléctrico mensual
# ============================================================
fig, ax = plt.subplots(figsize=(14, 6))
df_cons = df[df['consumo_fose'].notna() & (df['consumo_fose'] > 0)].copy()
bins = np.arange(55, 226, 5)
eleg = df_cons[(df_cons['consumo_fose'] >= 55) & (df_cons['consumo_fose'] < 130)]
tramo = df_cons[(df_cons['consumo_fose'] >= 130) & (df_cons['consumo_fose'] < 140)]
excl = df_cons[(df_cons['consumo_fose'] >= 140) & (df_cons['consumo_fose'] <= 225)]
ax.hist(eleg['consumo_fose'], bins=bins, weights=eleg['Fact_exp_hog']/1000, color='#2B6E99', edgecolor='white', linewidth=0.5, label='Elegibles (≤140 kWh)')
ax.hist(tramo['consumo_fose'], bins=bins, weights=tramo['Fact_exp_hog']/1000, color='#C44E52', edgecolor='white', linewidth=0.5, label='Tramo [130-140) kWh')
ax.hist(excl['consumo_fose'], bins=bins, weights=excl['Fact_exp_hog']/1000, color='#808080', edgecolor='white', linewidth=0.5, label='Excluidos (>140 kWh)')
ax.axvline(x=140, color='black', linestyle='--', linewidth=2)
ax.text(141, ax.get_ylim()[1]*0.9, 'Umbral FOSE\n140 kWh', fontsize=11, fontweight='bold')
hog_130_140 = tramo['Fact_exp_hog'].sum() / 1000
ax.annotate(f'[130-140): {hog_130_140:.0f}k hog.\nRatio pond. = 0.81\n(sin bunching neto)', xy=(135, hog_130_140*0.5), fontsize=9, color='#C44E52', bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor='#C44E52'))
ax.set_xlabel('Consumo eléctrico mensual (kWh)', fontsize=12)
ax.set_ylabel('Hogares representados (miles)', fontsize=12)
ax.legend(fontsize=10, loc='upper right')
fig.suptitle('Figura 1. Distribución del consumo eléctrico mensual y umbral de elegibilidad FOSE\n(ERCUE 2025; ponderado por factor de expansión; bins de 5 kWh)', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('fig1.png', dpi=300, bbox_inches='tight')
plt.close()
print("Fig 1 OK")

# ============================================================
# FIGURA 2: Energy burden por decil + tasa 2M por sector
# ============================================================
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5))
df_eb = df[df['energy_burden'].notna() & (df['energy_burden'] > 0) & (df['energy_burden'] < 1)].copy()
df_eb['decil'] = pd.qcut(df_eb['ga_aj_total_pc'], 10, labels=range(1, 11))
eb_decil = df_eb.groupby('decil')['energy_burden'].mean() * 100
ax1.bar(range(1, 11), eb_decil.values, color='#2B6E99', width=0.7)
ax1.axhline(y=6.26, color='#C44E52', linestyle=':', linewidth=1.5)
ax1.text(8.5, 6.6, '6,26%', fontsize=9, color='#C44E52')
ax1.set_xlabel('Decil de gasto per cápita\n(1=más pobre, 10=más rico)', fontsize=11)
ax1.set_ylabel('Energy burden promedio (%)', fontsize=11)
ax1.set_title('Panel A. Energy burden por decil de gasto', fontsize=12, fontweight='bold')
ax1.set_xticks(range(1, 11))
df_st = df[df['sobrecarga_2m'].notna() & df['sector_tipico'].notna()].copy()
sectores = ['ST1', 'ST2', 'ST3', 'ST4', 'SER']
tasas = [(df_st[df_st['sector_tipico']==st]['sobrecarga_2m'].mean()*100) for st in sectores]
colores = ['#2B6E99', '#4A8DB7', '#808080', '#808080', '#C44E52']
bars = ax2.bar(sectores, tasas, color=colores, width=0.6)
for bar, tasa in zip(bars, tasas):
    ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3, f'{tasa:.1f}%', ha='center', fontsize=10, fontweight='bold')
ax2.set_xlabel('Sector típico', fontsize=11)
ax2.set_ylabel('Tasa de sobrecarga energética 2M (%)', fontsize=11)
ax2.set_title('Panel B. Tasa 2M por sector típico', fontsize=12, fontweight='bold')
fig.suptitle('Figura 2. Energy burden y sobrecarga energética por decil de gasto y sector típico\n(ERCUE 2025; umbral 2M = 6,26%)', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('fig2.png', dpi=300, bbox_inches='tight')
plt.close()
print("Fig 2 OK")

# ============================================================
# FIGURA 3: Errores de focalización
# ============================================================
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(16, 5.5))
df_cc = df[df['consumo_fose'].notna() & df['sector_tipico'].notna() & df['elegible'].notna()].copy()
ei_v, ee_v, mu_v = [], [], []
for st in sectores:
    sub = df_cc[df_cc['sector_tipico']==st]
    eleg = sub[sub['elegible']==1]
    pob = sub[sub['pobre']==True]
    ei_v.append((eleg['pobre']==False).sum()/len(eleg)*100 if len(eleg)>0 else 0)
    ee_v.append((pob['elegible']==0).sum()/len(pob)*100 if len(pob)>0 else 0)
    mu_v.append(eleg['multipredio'].sum()/len(eleg)*100 if len(eleg)>0 else 0)
for ax, vals, color, titulo in [(ax1,ei_v,'#E8713A','Error de Inclusión (EI)\n% elegibles no pobres'), (ax2,ee_v,'#C44E52','Error de Exclusión (EE)\n% pobres excluidos por consumo'), (ax3,mu_v,'#2B6E99','Multipredio\n% elegibles con suministro compartido')]:
    bars = ax.bar(sectores, vals, color=color, width=0.6)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3, f'{val:.1f}%', ha='center', fontsize=10, fontweight='bold')
    ax.set_title(titulo, fontsize=11, fontweight='bold')
fig.suptitle('Figura 3. Errores de focalización del FOSE por sector típico (ERCUE 2025)', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('fig3.png', dpi=300, bbox_inches='tight')
plt.close()
print("Fig 3 OK")

# ============================================================
# FIGURA 4: Tasa 2M por tramo de consumo
# ============================================================
fig, ax = plt.subplots(figsize=(14, 6))
df_t = df[df['consumo_fose'].notna() & df['sobrecarga_2m'].notna()].copy()
bins_edges = np.arange(60, 230, 10)
for i in range(len(bins_edges)-1):
    low, high = bins_edges[i], bins_edges[i+1]
    sub = df_t[(df_t['consumo_fose']>=low)&(df_t['consumo_fose']<high)]
    if len(sub)>0:
        tasa = sub['sobrecarga_2m'].mean()*100
        color = '#2B6E99' if high<=140 else '#808080'
        ax.bar((low+high)/2, tasa, width=8, color=color, edgecolor='white')
ax.axvline(x=140, color='black', linestyle='--', linewidth=2)
ax.axhline(y=6.26, color='#C44E52', linestyle=':', linewidth=1.5)
ax.text(62, 6.6, 'Umbral 2M (6,26%)', fontsize=9, color='#C44E52')
ax.axvspan(120, 160, alpha=0.08, color='black')
ax.text(130, 3, '±20 kWh\nventana', fontsize=9, color='gray', ha='center')
ax.set_xlabel('Consumo eléctrico mensual (kWh)', fontsize=12)
ax.set_ylabel('Tasa de sobrecarga energética 2M (%)', fontsize=12)
fig.suptitle('Figura 4. Tasa de sobrecarga energética (indicador 2M) por tramo de consumo\n(ERCUE 2025; bins de 10 kWh)', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('fig4.png', dpi=300, bbox_inches='tight')
plt.close()
print("Fig 4 OK")

# ============================================================
# FIGURAS 5, 6, 7, 10: Solo usan números hardcodeados
# ============================================================

# FIGURA 5: Forest plot
fig, ax = plt.subplots(figsize=(12, 6))
ests = [('CEM ±20 kWh\n(principal)', -2.37, 2.43, 0.330), ('CEM ±30 kWh\n(robustez)', -5.72, 1.94, 0.003), ('IPW ±20 kWh\n(robustez)', -2.09, 2.35, 0.376), ('Sin matching\n±20 kWh', -1.03, None, None)]
for i, (label, att, se, pval) in enumerate(ests):
    y = 3-i
    color = '#C44E52' if pval and pval<0.10 else '#2B6E99'
    marker = 's' if pval and pval<0.10 else 'o'
    ax.plot(att, y, marker=marker, markersize=12, color=color, zorder=5)
    if se:
        ax.plot([att-1.96*se, att+1.96*se], [y,y], color=color, linewidth=1.5, alpha=0.4)
        ax.plot([att-1.645*se, att+1.645*se], [y,y], color=color, linewidth=4, alpha=0.5)
        sig = 'n.s.' if pval>=0.10 else f'p={pval:.3f}'
        ax.text(att+1.96*se+0.3, y, sig, fontsize=10, va='center', color=color)
ax.axvline(x=0, color='black', linewidth=1)
ax.axvspan(-2, 2, alpha=0.05, color='gray')
ax.set_yticks([3,2,1,0])
ax.set_yticklabels([e[0] for e in ests], fontsize=11)
ax.set_xlabel('Diferencia en tasa de sobrecarga 2M (pp)', fontsize=11)
legend_elements = [Line2D([0],[0],marker='o',color='w',markerfacecolor='#2B6E99',markersize=10,label='p ≥ 0,10'), Line2D([0],[0],marker='s',color='w',markerfacecolor='#C44E52',markersize=10,label='p < 0,10')]
ax.legend(handles=legend_elements, loc='lower left', fontsize=9)
fig.suptitle('Figura 5. Diferencias en sobrecarga energética (2M) asociadas a la elegibilidad FOSE\nComparación de estimadores', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('fig5.png', dpi=300, bbox_inches='tight')
plt.close()
print("Fig 5 OK")

# FIGURA 6: Sensibilidad bandwidth
fig, ax = plt.subplots(figsize=(12, 6))
bws = [10,15,20,25,30,35]
atts = [-5.85,-4.61,-2.37,-3.89,-5.72,-6.40]
ses = [3.51,2.86,2.43,2.02,1.94,1.86]
pvals = [0.096,0.107,0.330,0.054,0.003,0.001]
ns = [578,920,1284,1648,1945,2227]
for i in range(len(bws)):
    color = '#C44E52' if pvals[i]<0.10 else '#2B6E99'
    ax.plot([i,i], [atts[i]-1.96*ses[i], atts[i]+1.96*ses[i]], color='#AAAAAA', linewidth=1.5)
    ax.plot([i,i], [atts[i]-1.645*ses[i], atts[i]+1.645*ses[i]], color=color, linewidth=4, alpha=0.6)
    ax.plot(i, atts[i], marker='s' if pvals[i]<0.10 else 'o', markersize=12, color=color, zorder=5)
    ax.text(i, atts[i]+1.96*ses[i]+0.5, f'N={ns[i]}\np={pvals[i]:.3f}', ha='center', fontsize=8, color='gray')
ax.axhline(y=0, color='black', linestyle='--', linewidth=1)
ax.set_xticks(range(len(bws)))
ax.set_xticklabels([f'±{b}' for b in bws], fontsize=12)
ax.set_xlabel('Semiancho de ventana (kWh)', fontsize=12)
ax.set_ylabel('ATT (puntos porcentuales)', fontsize=12)
fig.suptitle('Figura 6. Sensibilidad del ATT al ancho de ventana (CEM)', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('fig6.png', dpi=300, bbox_inches='tight')
plt.close()
print("Fig 6 OK")

# FIGURA 7: Balance SMD
fig, ax = plt.subplots(figsize=(10, 7))
variables = ['Sector ST3-ST4','Hogar 5-6 pers.','Sector ST1-ST2','Hogar 7+ pers.','Sector SER','Área rural','Área urbana','Hogar 3-4 pers.','Hogar 1-2 pers.','Tamaño hogar (cont.)']
smd_a = [0.0446,0.0079,0.0095,0.0307,0.0569,0.0341,0.0341,0.0763,0.0829,0.0947]
smd_d = [0.0447,0.0081,0.0138,0.0230,0.0495,0.0479,0.0479,0.0796,0.0833,0.0900]
for i in range(len(variables)):
    ax.plot([smd_a[i],smd_d[i]], [i,i], color='gray', linewidth=1, zorder=1)
ax.scatter(smd_a, range(len(variables)), color='#C44E52', s=80, zorder=3, label='Antes del CEM')
ax.scatter(smd_d, range(len(variables)), color='#2B6E99', s=100, marker='D', zorder=3, label='Después del CEM')
ax.axvline(x=0.10, color='gray', linestyle='--', linewidth=1.5)
ax.set_yticks(range(len(variables)))
ax.set_yticklabels(variables, fontsize=11)
ax.set_xlabel('|SMD|', fontsize=12)
ax.set_xlim(-0.005, 0.12)
ax.legend(fontsize=10, loc='lower right')
fig.suptitle('Figura 7. Balance de covariables antes y después del CEM\n(ventana ±20 kWh; línea punteada = umbral |SMD| = 0,10)', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('fig7.png', dpi=300, bbox_inches='tight')
plt.close()
print("Fig 7 OK")

# ============================================================
# FIGURA 8: Overlap propensity score
# ============================================================
from sklearn.linear_model import LogisticRegression
df_cem = df[df['consumo_fose'].notna() & df['sector_tipico'].notna() & df['sobrecarga_2m'].notna() & df['elegible'].notna()].copy()
df_cem['T'] = df_cem['elegible'].astype(int)
df_cem['tamano_cat'] = pd.cut(df_cem['N_PERSONAS'], bins=[0,2,4,6,99], labels=['1-2','3-4','5-6','7+'])
df_cem['sector_cem'] = df_cem['sector_tipico'].replace({'ST1':'ST1-ST2','ST2':'ST1-ST2','ST3':'ST3-ST4','ST4':'ST3-ST4'})
v20 = df_cem[(df_cem['consumo_fose']>=120)&(df_cem['consumo_fose']<=160)].copy()
v20['celda'] = v20['sector_cem'].astype(str)+'_'+v20['area'].astype(str)+'_'+v20['tamano_cat'].astype(str)
celdas = v20.groupby('celda')['T'].agg(['sum','count'])
celdas['nc'] = celdas['count']-celdas['sum']
celdas = celdas[(celdas['sum']>0)&(celdas['nc']>0)]
vm = v20[v20['celda'].isin(celdas.index)].copy()
X = pd.get_dummies(vm[['sector_cem','area','tamano_cat']], drop_first=True).astype(float)
lr = LogisticRegression(max_iter=1000)
lr.fit(X, vm['T'].values)
vm['ps'] = lr.predict_proba(X)[:,1]
fig, ax = plt.subplots(figsize=(10, 6))
bins_ps = np.linspace(0, 1, 50)
ax.hist(vm.loc[vm['T']==1,'ps'], bins=bins_ps, alpha=0.5, color='#2B6E99', density=True, label='Elegibles (≤140 kWh)')
ax.hist(vm.loc[vm['T']==0,'ps'], bins=bins_ps, alpha=0.5, color='#C44E52', density=True, label='Excluidos (>140 kWh)')
ax.axvline(x=vm.loc[vm['T']==1,'ps'].mean(), color='#2B6E99', linestyle='--', linewidth=2, label=f'Media elegibles: {vm.loc[vm["T"]==1,"ps"].mean():.3f}')
ax.axvline(x=vm.loc[vm['T']==0,'ps'].mean(), color='#C44E52', linestyle='-.', linewidth=2, label=f'Media excluidos: {vm.loc[vm["T"]==0,"ps"].mean():.3f}')
ax.set_xlabel('Propensity Score estimado', fontsize=12)
ax.set_ylabel('Densidad', fontsize=12)
ax.legend(fontsize=10)
fig.suptitle('Figura 8. Distribución del propensity score por grupo\n(ventana ±20 kWh; logit con sector típico, área y tamaño del hogar)', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('fig8.png', dpi=300, bbox_inches='tight')
plt.close()
print("Fig 8 OK")

# ============================================================
# DESCARGAR TODOS
# ============================================================
import glob
for f in sorted(glob.glob('fig*.png')):
    print(f"Descargando {f}...")
    files.download(f)
print("\nTODOS DESCARGADOS")

In [ ]:
import glob
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from google.colab import files

# Buscar el Excel
archivos = glob.glob('/Evaluaci*') + glob.glob('Evaluaci*')
print(f"Encontrado: {archivos}")

fose_excel = pd.read_excel(archivos[0], sheet_name='FOSE', header=7)

# GRÁFICO 20 (versión alternativa): Simulación contrafactual
tarifas_cf = fose_excel[['Código Sistema','De 0 a 30 kw','De 30 a 140 kw','Más de 140 kw']].copy()
tarifas_cf.columns = ['sistema','tarifa_cf_b1','tarifa_cf_b2','tarifa_cf_b3']
tarifas_cf['sistema'] = tarifas_cf['sistema'].str.strip()
tarifas_sf = fose_excel[['Código Sistema','De 0 a 30 kw.2','De 30 a 140 kw.2','Más de 140 kw.2']].copy()
tarifas_sf.columns = ['sistema','tarifa_sf_b1','tarifa_sf_b2','tarifa_sf_b3']
tarifas_sf['sistema'] = tarifas_sf['sistema'].str.strip()
sim = df[df['consumo_fose'].notna() & df['gasto_mes_electricidad'].notna() & df['sector_tipico'].notna() & (df['consumo_fose']>0) & (df['elegible']==1)].copy()
sim['sistema'] = sim['sistema'].str.strip()
sim = sim.merge(tarifas_cf, on='sistema', how='left').merge(tarifas_sf, on='sistema', how='left')
b1 = sim['consumo_fose'].clip(upper=30)
b2 = (sim['consumo_fose']-30).clip(lower=0, upper=110)
sim['fac_con'] = b1*sim['tarifa_cf_b1']+b2*sim['tarifa_cf_b2']
sim['fac_sin'] = b1*sim['tarifa_sf_b1']+b2*sim['tarifa_sf_b2']
sim['descuento'] = sim['fac_sin']-sim['fac_con']
sim['gasto_sin_fose'] = sim['gasto_mes_electricidad']+sim['descuento']
gt = sim['ga_aj_total_pc']*sim['N_PERSONAS']
sim['eb_con'] = sim['gasto_mes_electricidad']/gt
sim['eb_sin'] = sim['gasto_sin_fose']/gt
umbral = 0.0626
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5))
sv = sim[sim['eb_con'].notna()&sim['eb_sin'].notna()&(sim['eb_con']<0.5)&(sim['eb_sin']<0.5)]
ax1.hist(sv['eb_con']*100, bins=50, alpha=0.5, color='#2B6E99', density=True, label='Con FOSE')
ax1.hist(sv['eb_sin']*100, bins=50, alpha=0.5, color='#C44E52', density=True, label='Sin FOSE')
ax1.axvline(x=6.26, color='black', linestyle='--', linewidth=1.5, label='Umbral 2M (6,26%)')
ax1.set_xlabel('Energy burden (%)', fontsize=11)
ax1.set_ylabel('Densidad', fontsize=11)
ax1.set_title('A. Distribución del energy burden', fontsize=11, fontweight='bold')
ax1.legend(fontsize=9)
sects = ['ST1','ST2','ST3','ST4','SER']
cv, sv2 = [], []
for st in sects:
    s = sim[sim['sector_tipico']==st]
    cv.append((s['eb_con']>umbral).mean()*100)
    sv2.append((s['eb_sin']>umbral).mean()*100)
x = np.arange(len(sects))
w = 0.35
b1p = ax2.bar(x-w/2, cv, w, color='#2B6E99', label='Con FOSE')
b2p = ax2.bar(x+w/2, sv2, w, color='#C44E52', label='Sin FOSE')
for bar,val in zip(b1p,cv): ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3, f'{val:.1f}%', ha='center', fontsize=9, color='#2B6E99')
for bar,val in zip(b2p,sv2): ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3, f'{val:.1f}%', ha='center', fontsize=9, color='#C44E52')
ax2.set_xticks(x)
ax2.set_xticklabels(sects, fontsize=11)
ax2.set_xlabel('Sector típico', fontsize=11)
ax2.set_ylabel('Tasa de sobrecarga 2M (%)', fontsize=11)
ax2.set_title('B. Tasa 2M por sector típico', fontsize=11, fontweight='bold')
ax2.legend(fontsize=10)
fig.suptitle('Figura 9. Simulación contrafactual: energy burden con y sin FOSE\n(ERCUE 2025; hogares elegibles ≤140 kWh)', fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('fig9.png', dpi=300, bbox_inches='tight')
plt.close()
print("Fig 9 OK")

# FIGURA 10
fig, ax = plt.subplots(figsize=(10, 6))
cats = ['No pobre\n(N=1.132)','Pobre no extremo\n(N=121)','Pobre extremo\n(N=9)']
atts = [-0.23, -20.29, None]
ses = [2.19, 8.22, None]
pvs = [0.918, 0.014, None]
for i in range(3):
    y = 2-i
    if atts[i] is None:
        ax.plot(0, y, marker='x', markersize=14, color='gray', zorder=5)
        ax.text(1, y, 'No estimable\n(N=9, sin varianza)', fontsize=10, va='center', color='gray', style='italic')
        continue
    color = '#C44E52' if pvs[i]<0.05 else '#2B6E99'
    ax.plot(atts[i], y, marker='s' if pvs[i]<0.05 else 'o', markersize=14, color=color, zorder=5)
    ax.plot([atts[i]-1.96*ses[i], atts[i]+1.96*ses[i]], [y,y], color=color, linewidth=1.5, alpha=0.4)
    ax.plot([atts[i]-1.645*ses[i], atts[i]+1.645*ses[i]], [y,y], color=color, linewidth=4, alpha=0.5)
    sig = f'p={pvs[i]:.3f}' if pvs[i]<0.05 else 'n.s.'
    ax.text(atts[i]+1.96*ses[i]+0.5, y, sig, fontsize=11, va='center', color=color, fontweight='bold')
ax.axvline(x=0, color='black', linewidth=1)
ax.set_yticks([2,1,0])
ax.set_yticklabels(cats, fontsize=12)
ax.set_xlabel('ATT: diferencia en tasa de sobrecarga 2M (pp)', fontsize=11)
ax.annotate('Brecha de 20 pp\n(evidencia exploratoria)', xy=(-20.29,1), xytext=(-35,1.6), fontsize=10, color='#C44E52', arrowprops=dict(arrowstyle='->', color='#C44E52', linewidth=1.5))
fig.suptitle('Figura 10. Heterogeneidad por condición de pobreza (CEM ±20 kWh)\nHallazgo robusto en pobres no extremos (sensibilidad por bandwidth)', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('fig10.png', dpi=300, bbox_inches='tight')
plt.close()
print("Fig 10 OK")

# Descargar
for f in ['fig9.png', 'fig10.png']:
    files.download(f)
print("DESCARGADOS")

In [ ]:
# ============================================================
# GRÁFICO 19: Sensibilidad por bandwidth — Pobres no extremos
# ============================================================
# Verifica si el hallazgo de −20 pp en pobres no extremos
# es robusto o depende de la ventana elegida.
# Resultado: se mantiene entre −21 y −31 pp en todas las ventanas.
# ============================================================

fig, ax = plt.subplots(figsize=(10, 6))

bws = [15, 20, 25, 30]
atts = [-21.81, -20.29, -26.86, -30.58]
ses = [11.50, 8.22, 7.03, 6.33]
pvals = [0.058, 0.014, 0.000, 0.000]
ns = [76, 121, 165, 199]

for i in range(len(bws)):
    color = '#C44E52' if pvals[i] < 0.10 else '#2B6E99'
    ax.plot([i, i], [atts[i]-1.96*ses[i], atts[i]+1.96*ses[i]], color='#AAAAAA', linewidth=1.5)
    ax.plot([i, i], [atts[i]-1.645*ses[i], atts[i]+1.645*ses[i]], color=color, linewidth=4, alpha=0.6)
    ax.plot(i, atts[i], marker='s', markersize=12, color=color, zorder=5)
    ax.text(i, atts[i]+1.96*ses[i]+1.5, f'N={ns[i]}\np={pvals[i]:.3f}', ha='center', fontsize=9, color='gray')

ax.axhline(y=0, color='black', linestyle='--', linewidth=1)
ax.set_xticks(range(len(bws)))
ax.set_xticklabels([f'±{b}' for b in bws], fontsize=12)
ax.set_xlabel('Semiancho de ventana (kWh)', fontsize=12)
ax.set_ylabel('ATT (puntos porcentuales)', fontsize=12)

fig.suptitle('Figura 11. Sensibilidad por bandwidth: pobres no extremos\nEl hallazgo de −20 pp se mantiene en todas las ventanas', fontsize=13, fontweight='bold', y=1.02)

plt.tight_layout()
plt.savefig('fig11.png', dpi=300, bbox_inches='tight')
plt.close()
print("Fig 11 OK")

files.download('fig11.png')
print("DESCARGADO")